# External experimental validation of PDAC drug-combination models

This notebook evaluates the transferability of three retained
classifiers (HistGradientBoosting, LightGBM and CatBoost) to five
experimental drug-combination datasets.

The models are refitted on 8,450 internal training units using
symmetric Morgan fingerprints and evaluated separately for
two training endpoints and three chemical-novelty categories.

## 0. Configuration and repository path discovery

In [1]:
from __future__ import annotations
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display
import json, re, hashlib, sys, platform, warnings
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.ensemble import HistGradientBoostingClassifier
import sklearn
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from rdkit import Chem, DataStructs, rdBase
from rdkit.Chem import AllChem
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# EDIT THESE ONLY WHEN NECESSARY.
PROJECT_ROOT_OVERRIDE = None
TRAINING_UNITS_PATH = None  # default: versioned GitHub CSV, else original canonical .pkl
NCATS_RAW_PATH = None
NOTEBOOK40_CONTRACT_PATH = 'results/tables/ml_comprehensive/external_dataset_harmonisation/external_source_harmonisation_contract.csv'
PUBCHEM_CACHE_PATH = 'data/processed/external_validation/external_pubchem_smiles_cache.csv'

EXPECTED_N_UNITS = 8450
EXPECTED_ANY_POSITIVES = 1063
EXPECTED_STRICT_MAJORITY_POSITIVES = 188
STRICT_MAJORITY_THRESHOLD = 0.5

EXPECTED_SOURCE_SIZES = {'Yale':820,'NCATS':496,'Hoefer':146,'Vis':48,'Bashi':109}
FINAL_MODEL_PANEL = {'hist_gradient_boosting','lightgbm_gbdt','catboost'}
ENDPOINTS = {'any_positive_label','majority_label'}

def find_project_root():
    if PROJECT_ROOT_OVERRIDE is not None:
        p=Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        if not (p/'data').is_dir(): raise FileNotFoundError(f'PROJECT_ROOT_OVERRIDE missing data/: {p}')
        return p
    start=Path.cwd().resolve()
    for p in (start,*start.parents):
        if (p/'data').is_dir() and ((p/'.git').exists() or (p/'notebooks').is_dir()): return p
    raise FileNotFoundError('Run from the repository or set PROJECT_ROOT_OVERRIDE to its root.')
ROOT=find_project_root()
def repo_path(value):
    p=Path(value).expanduser()
    return (p if p.is_absolute() else ROOT/p).resolve()
def rel(path):
    p=Path(path).resolve()
    return p.relative_to(ROOT).as_posix() if p.is_relative_to(ROOT) else str(p)
def sha256_file(path,block=1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(block),b''):h.update(chunk)
    return h.hexdigest()
def as_bool_series(s):
    s=pd.Series(s)
    if pd.api.types.is_bool_dtype(s):return s.fillna(False)
    return s.astype(str).str.strip().str.lower().map({'true':True,'false':False,'1':True,'0':False}).fillna(False).astype(bool)

RESULTS_ROOT = (
    ROOT / "results" / "tables" / "ml_comprehensive"
)

OUT = (
    RESULTS_ROOT
    / "external_experimental_validation"
)

DERIVED_OUT = OUT

FIG = (
    ROOT
    / "results"
    / "figures"
    / "ml_comprehensive"
    / "external_experimental_validation"
)

for path in (OUT, DERIVED_OUT, FIG):
    path.mkdir(parents=True, exist_ok=True)

SOURCE_CONTRACT=repo_path(NOTEBOOK40_CONTRACT_PATH)
CACHE_PATH=repo_path(PUBCHEM_CACHE_PATH)
pd.set_option('display.max_columns',200)
pd.set_option('display.width',220)
print('Repository:',ROOT)
print('Notebook 40 contract:',rel(SOURCE_CONTRACT))
print('External validation output:',rel(OUT))


Repository: C:\Users\Käyttäjä\Documents\PDAC-drug-combination-ML-MSc-thesis
Notebook 40 contract: results/tables/ml_comprehensive/external_dataset_harmonisation/external_source_harmonisation_contract.csv
External validation output: results/tables/ml_comprehensive/external_experimental_validation


## 1. Internal Jaaks/GDSC training data and endpoint definitions

In [2]:
TRAIN_PATH = repo_path(
    TRAINING_UNITS_PATH
    or "data/processed/final_github/"
       "canonical_order_invariant_unit_with_morgan_github.csv"
)

if not TRAIN_PATH.is_file():
    raise FileNotFoundError(
        f"Required canonical training CSV missing: {TRAIN_PATH}"
    )

train = pd.read_csv(TRAIN_PATH, low_memory=False)

print(
    "Training source:", rel(TRAIN_PATH),
    "sha256:", sha256_file(TRAIN_PATH),
)
print("Training shape:", train.shape)

Training source: data/processed/final_github/canonical_order_invariant_unit_with_morgan_github.csv sha256: a96b262fdcc26f0b51be978194aac488b98219e1a4b17e38e58020328f20d125
Training shape: (8450, 1080)


In [3]:
if "positive_fraction" not in train.columns:
    raise KeyError("Training table does not contain positive_fraction.")

pf = pd.to_numeric(train["positive_fraction"], errors="coerce")
if pf.isna().any():
    raise ValueError("positive_fraction contains missing/non-numeric values.")

train = train.copy()
train["any_positive_label_FINAL"] = (pf > 0).astype(int)
train["majority_label_FINAL"] = (pf > STRICT_MAJORITY_THRESHOLD).astype(int)

endpoint_audit = pd.DataFrame([
    {
        "endpoint": "any_positive_label",
        "definition": "positive_fraction > 0",
        "n_units": len(train),
        "positive_count": int(train["any_positive_label_FINAL"].sum()),
        "expected_positive_count": EXPECTED_ANY_POSITIVES,
    },
    {
        "endpoint": "majority_label",
        "definition": "positive_fraction > 0.5 (STRICT)",
        "n_units": len(train),
        "positive_count": int(train["majority_label_FINAL"].sum()),
        "expected_positive_count": EXPECTED_STRICT_MAJORITY_POSITIVES,
    },
])

display(endpoint_audit)

assert len(train) == EXPECTED_N_UNITS, (len(train), EXPECTED_N_UNITS)
assert int(train["any_positive_label_FINAL"].sum()) == EXPECTED_ANY_POSITIVES
assert int(train["majority_label_FINAL"].sum()) == EXPECTED_STRICT_MAJORITY_POSITIVES

cols = list(train.columns)

# ============================================================
# IMPORTANT:
# The canonical Jaaks table stores the original 512-bit drug
# fingerprints as:
#
#   drug_a_fp_0 ... drug_a_fp_511
#   drug_b_fp_0 ... drug_b_fp_511
#
# They do NOT contain the word "morgan" in the column name.
# This is the exact schema used in the final cold-drug analysis.
# ============================================================

def fp_index(c):
    m = re.search(r"(\d+)$", str(c))
    return int(m.group(1)) if m else -1

drug_a_fp = sorted(
    [
        c for c in cols
        if re.fullmatch(r"drug_a_fp_\d+", str(c))
    ],
    key=fp_index,
)

drug_b_fp = sorted(
    [
        c for c in cols
        if re.fullmatch(r"drug_b_fp_\d+", str(c))
    ],
    key=fp_index,
)

# Compatibility fallbacks only if a future canonical table uses
# an explicitly named Morgan schema.
if len(drug_a_fp) != 512 or len(drug_b_fp) != 512:
    drug_a_fp = sorted(
        [
            c for c in cols
            if re.fullmatch(
                r"(?:drug_a|anchor)_(?:morgan_)?fp_\d+",
                str(c),
                flags=re.I,
            )
        ],
        key=fp_index,
    )
    drug_b_fp = sorted(
        [
            c for c in cols
            if re.fullmatch(
                r"(?:drug_b|library)_(?:morgan_)?fp_\d+",
                str(c),
                flags=re.I,
            )
        ],
        key=fp_index,
    )

print(
    "Stored fingerprint columns:",
    f"A={len(drug_a_fp)}, B={len(drug_b_fp)}",
)

if drug_a_fp:
    print(
        "A fingerprint range:",
        drug_a_fp[0],
        "...",
        drug_a_fp[-1],
    )
if drug_b_fp:
    print(
        "B fingerprint range:",
        drug_b_fp[0],
        "...",
        drug_b_fp[-1],
    )

if len(drug_a_fp) != 512 or len(drug_b_fp) != 512:
    fp_like = [
        c for c in cols
        if "fp" in str(c).lower()
        or "morgan" in str(c).lower()
    ]

    raise ValueError(
        "Could not recover the exact 512-bit fingerprints from "
        "the canonical training table. "
        f"Found A={len(drug_a_fp)}, B={len(drug_b_fp)}. "
        f"Example fingerprint-like columns: {fp_like[:20]}"
    )

# Hard ordering check: exact bit indices 0..511
a_indices = [fp_index(c) for c in drug_a_fp]
b_indices = [fp_index(c) for c in drug_b_fp]

if a_indices != list(range(512)):
    raise ValueError(
        "Drug A fingerprint columns are not exactly bits 0..511."
    )

if b_indices != list(range(512)):
    raise ValueError(
        "Drug B fingerprint columns are not exactly bits 0..511."
    )

A = (
    train[drug_a_fp]
    .apply(pd.to_numeric, errors="coerce")
    .to_numpy(dtype=np.float32)
)

B = (
    train[drug_b_fp]
    .apply(pd.to_numeric, errors="coerce")
    .to_numpy(dtype=np.float32)
)

if np.isnan(A).any() or np.isnan(B).any():
    raise ValueError(
        "NaN values found in stored training fingerprints."
    )

Xtrain = np.concatenate(
    [
        (A + B) / 2.0,
        np.abs(A - B),
    ],
    axis=1,
).astype(np.float32)

if Xtrain.shape != (
    EXPECTED_N_UNITS,
    1024,
):
    raise ValueError(
        f"Unexpected symmetric Morgan matrix shape: "
        f"{Xtrain.shape}"
    )

print("Final endpoint contract PASSED.")
print("Symmetric Morgan matrix:", Xtrain.shape)


             endpoint                        definition  n_units  positive_count  expected_positive_count
0  any_positive_label             positive_fraction > 0     8450            1063                     1063
1      majority_label  positive_fraction > 0.5 (STRICT)     8450             188                      188

Stored fingerprint columns: A=512, B=512
A fingerprint range: drug_a_fp_0000 ... drug_a_fp_0511
B fingerprint range: drug_b_fp_0000 ... drug_b_fp_0511
Final endpoint contract PASSED.
Symmetric Morgan matrix: (8450, 1024)


## 2. Canonical drug names and the complete internal familiarity space

In [4]:
ALIASES = {
    "5FU": "5FLUOROURACIL",
    "5FLUOROURACIL": "5FLUOROURACIL",
    "FLUOROURACIL": "5FLUOROURACIL",
    "MK1775": "ADAVOSERTIB",
    "AZD1775": "ADAVOSERTIB",
    "ADAVOSERTIB": "ADAVOSERTIB",
    "SCH727965": "DINACICLIB",
    "DINACICLIBSCH727965": "DINACICLIB",
    "DINACICLIB": "DINACICLIB",
    "GSK1120212": "TRAMETINIB",
    "TRAMETINIB": "TRAMETINIB",
    "AZD2281": "OLAPARIB",
    "OLAPARIB": "OLAPARIB",
    "OSI906": "LINSITINIB",
    "LINSITINIB": "LINSITINIB",
    "PF02341066": "CRIZOTINIB",
    "CRIZOTINIB": "CRIZOTINIB",
    "BMS354825": "DASATINIB",
    "DASATINIB": "DASATINIB",
    "GDC0032": "TASELISIB",
    "TASELISIB": "TASELISIB",
    "GDC0068": "IPATASERTIB",
    "IPATASERTIB": "IPATASERTIB",
    "BKM120": "BUPARLISIB",
    "BUPARLISIB": "BUPARLISIB",
    "PD0332991": "PALBOCICLIB",
    "PALBOCICLIB": "PALBOCICLIB",
    "INCB018424": "RUXOLITINIB",
    "RUXOLITINIB": "RUXOLITINIB",
    "LY2157299": "GALUNISERTIB",
    "GALUNISERTIB": "GALUNISERTIB",
    "BAY439006": "SORAFENIB",
    "SORAFENIB": "SORAFENIB",
    "ABT263": "NAVITOCLAX",
    "NAVITOCLAX": "NAVITOCLAX",
    "ABT199": "VENETOCLAX",
    "VENETOCLAX": "VENETOCLAX",
}

def canon_drug(x):
    if pd.isna(x):
        return None
    s = str(x).upper().strip()
    s = s.replace("−", "-").replace("–", "-")
    s = re.sub(r"\s*\(\s*-\s*\)\s*$", "", s)
    s = re.sub(r"[^A-Z0-9]+", "", s)
    if not s:
        return None
    return ALIASES.get(s, s)

def make_pair_key(a, b):
    a, b = canon_drug(a), canon_drug(b)
    if not a or not b:
        return None
    return "||".join(sorted([a, b]))

def find_name_col(candidates):
    for c in candidates:
        if c in train.columns:
            return c
    return None

PA = find_name_col(["drug_a_norm", "drug_a_canon", "drug_a", "ANCHOR_NAME", "anchor_name"])
PB = find_name_col(["drug_b_norm", "drug_b_canon", "drug_b", "LIBRARY_NAME", "library_name"])
if PA is None or PB is None:
    raise KeyError("Could not identify canonical drug-name columns in training table.")

internal_drugs = set()
internal_pairs = set()

for a, b in train[[PA, PB]].itertuples(index=False, name=None):
    ca, cb = canon_drug(a), canon_drug(b)
    if ca:
        internal_drugs.add(ca)
    if cb:
        internal_drugs.add(cb)
    pk = make_pair_key(ca, cb)
    if pk:
        internal_pairs.add(pk)

if len(internal_drugs) != 26 or len(internal_pairs) != 325:
    raise ValueError(
        f"Frozen internal space mismatch: drugs={len(internal_drugs)}, "
        f"pairs={len(internal_pairs)}; expected 26 and 325."
    )

print("Internal drugs:", len(internal_drugs))
print("Internal pairs:", len(internal_pairs))


Internal drugs: 26
Internal pairs: 325


## 3. Machine learning models

In [5]:

# Three retained model families, with the exact
# specifications stated in thesis Appendix B.

MODEL_SPEC_VERSION = "thesis_appendix_B_three_model_v1"

def make_model(name, y):
    if name == "hist_gradient_boosting":
        return HistGradientBoostingClassifier(
            max_iter=200,
            learning_rate=0.05,
            max_leaf_nodes=31,
            l2_regularization=1.0,
            early_stopping=True,
            random_state=42,
        )

    if name == "lightgbm_gbdt":
        return LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary",
            class_weight="balanced",
            verbosity=-1,
            n_jobs=-1,
            random_state=42,
        )

    if name == "catboost":
        return CatBoostClassifier(
            iterations=500,
            depth=4,
            learning_rate=0.05,
            loss_function="Logloss",
            eval_metric="AUC",
            auto_class_weights="Balanced",
            verbose=False,
            allow_writing_files=False,
            random_seed=42,
        )

    raise KeyError(f"Unexpected model: {name}")


MODELS = [
    "hist_gradient_boosting",
    "lightgbm_gbdt",
    "catboost",
]

TARGETS = {
    "any_positive_label":
        train["any_positive_label_FINAL"].to_numpy(int),
    "majority_label":
        train["majority_label_FINAL"].to_numpy(int),
}

assert set(MODELS) == FINAL_MODEL_PANEL
assert set(TARGETS) == ENDPOINTS

for endpoint, y in TARGETS.items():
    assert len(y) == 8450

for name in MODELS:
    params = make_model(name, TARGETS["any_positive_label"]).get_params()
    print(name, params)


hist_gradient_boosting {'categorical_features': 'from_dtype', 'class_weight': None, 'early_stopping': True, 'interaction_cst': None, 'l2_regularization': 1.0, 'learning_rate': 0.05, 'loss': 'log_loss', 'max_bins': 255, 'max_depth': None, 'max_features': 1.0, 'max_iter': 200, 'max_leaf_nodes': 31, 'min_samples_leaf': 20, 'monotonic_cst': None, 'n_iter_no_change': 10, 'random_state': 42, 'scoring': 'loss', 'tol': 1e-07, 'validation_fraction': 0.1, 'verbose': 0, 'warm_start': False}
lightgbm_gbdt {'boosting_type': 'gbdt', 'class_weight': 'balanced', 'colsample_bytree': 0.8, 'importance_type': 'split', 'learning_rate': 0.05, 'max_depth': -1, 'min_child_samples': 20, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 300, 'n_jobs': -1, 'num_leaves': 31, 'objective': 'binary', 'random_state': 42, 'reg_alpha': 0.0, 'reg_lambda': 0.0, 'subsample': 0.8, 'subsample_for_bin': 200000, 'subsample_freq': 0, 'verbosity': -1}
catboost {'iterations': 500, 'learning_rate': 0.05, 'depth': 

## 4. External experimental sources and input provenance

In [6]:
if not SOURCE_CONTRACT.is_file():
    raise FileNotFoundError(f'Run GitHub notebook 40 first; source contract missing: {SOURCE_CONTRACT}')
source_contract=pd.read_csv(SOURCE_CONTRACT)
required={'dataset','expected_pairs','binary_expected','continuous_expected','output_path'}
if not required.issubset(source_contract):
    raise KeyError(f'Notebook 40 contract columns missing: {sorted(required-set(source_contract.columns))}')
if len(source_contract)!=4 or set(source_contract.dataset)!={'Yale','Höfer','Vis','Bashi'}:
    raise AssertionError('Notebook 40 manifest must contain exactly Yale, Höfer, Vis and Bashi.')
source_paths={}
for row in source_contract.itertuples(index=False):
    p=repo_path(row.output_path)
    if not p.is_relative_to(ROOT):raise ValueError('Notebook 40 source lies outside GitHub repository: '+str(p))
    if not p.is_file():raise FileNotFoundError(f'Notebook 40 source {row.dataset} missing: {p}')
    raw=pd.read_csv(p,low_memory=False)
    part=raw.loc[raw.dataset.eq(row.dataset)] if row.dataset in {'Vis','Bashi'} else raw
    if len(part)!=int(row.expected_pairs):
        raise AssertionError(f'Notebook 40 {row.dataset}: {len(part)} rows; expected {row.expected_pairs}')
    if 'pair_key' not in part or part.pair_key.isna().any() or part.pair_key.nunique()!=len(part):
        raise AssertionError(f'Notebook 40 {row.dataset}: missing or duplicate pair keys')
    source_paths['Hoefer' if row.dataset=='Höfer' else row.dataset]=p
    print('Verified notebook 40:',row.dataset,len(part),'pairs',rel(p))

def locate_ncats_raw():
    if NCATS_RAW_PATH is not None:
        p=repo_path(NCATS_RAW_PATH)
        if not p.is_file():raise FileNotFoundError(f'NCATS_RAW_PATH not found: {p}')
        return p
    candidates=sorted(p for p in (ROOT/'data').rglob('Training_set.csv') if p.is_file())
    if len(candidates)!=1:
        raise FileNotFoundError(
            f'Expected exactly one source Training_set.csv, found {len(candidates)}: '
            +repr([rel(p) for p in candidates])+'. Configure NCATS_RAW_PATH explicitly.')
    return candidates[0]
NCATS_INPUT=locate_ncats_raw()
ncats_raw=pd.read_csv(NCATS_INPUT,low_memory=False)
expected={'Sample Name 1','Sample Name 2','Structure 1','Structure 2','Gamma','Label'}
if not expected.issubset(ncats_raw):
    raise KeyError(f'NCATS raw source is missing columns: {sorted(expected-set(ncats_raw.columns))}. Actual: {list(ncats_raw.columns)}')
if len(ncats_raw)!=496:
    raise AssertionError(f'Expected 496 NCATS source rows, got {len(ncats_raw)}')

def ncats_binary_labels(series):
    v=series.astype(str).str.strip().str.lower()
    mapped=v.map({'1':1,'0':0,'1.0':1,'0.0':0,'true':1,'false':0,
                  'synergistic':1,'non-synergistic':0,'synergy':1,'non synergy':0,'yes':1,'no':0})
    if mapped.isna().any():
        raise ValueError('NCATS original Label has unsupported values: '+str(v[mapped.isna()].unique()[:8]))
    if mapped.nunique()!=2:raise AssertionError('NCATS source has fewer than two label classes')
    return mapped.astype(int)

ncats_standardized=pd.DataFrame({
    'drug_a':ncats_raw['Sample Name 1'],
    'drug_b':ncats_raw['Sample Name 2'],
    'Gamma':pd.to_numeric(ncats_raw['Gamma'],errors='raise'),
    'label':ncats_binary_labels(ncats_raw['Label']),
})
if ncats_standardized[['drug_a','drug_b','Gamma','label']].isna().any().any():
    raise ValueError('Missing NCATS experimental response or compound name')
NCATS_ADAPTER=OUT/'ncats_from_raw_training_set.csv'
ncats_standardized.to_csv(NCATS_ADAPTER,index=False)
source_paths['NCATS']=NCATS_ADAPTER
print('NCATS raw source:',rel(NCATS_INPUT),'sha256:',sha256_file(NCATS_INPUT))
print('Fresh NCATS 496-pair adapter:',rel(NCATS_ADAPTER))


Verified notebook 40: Yale 820 pairs results/tables/ml_comprehensive/external_validation_yale_pdac/yale_pair_level_external_candidate_summary.csv
Verified notebook 40: Höfer 146 pairs results/tables/ml_comprehensive/external_hoefer_2025_cross_assay_validation/hoefer_inhibitor_pair_level_summary.csv
Verified notebook 40: Vis 48 pairs results/tables/ml_comprehensive/vis_bashi_independent_external_validation_final_strict/vis_bashi_harmonized_pair_level_test_units.csv
Verified notebook 40: Bashi 109 pairs results/tables/ml_comprehensive/vis_bashi_independent_external_validation_final_strict/vis_bashi_harmonized_pair_level_test_units.csv
NCATS raw source: data/raw/PANC1/Training_set.csv sha256: 055e7026b984c8e8f15bebb94fbbc65b5437529dd4b69811ae6c0e8941f49943
Fresh NCATS 496-pair adapter: results/tables/ml_comprehensive/external_experimental_validation/ncats_from_raw_training_set.csv


## 5. Harmonisation of study-specific experimental responses

In [7]:
SOURCE_REGISTRY = pd.DataFrame([
    {
        "dataset": "Yale",
        "search_tokens": "yale",
        "study_role": "multi-cell-line cross-assay external screen",
        "independence_class": "cross_assay_external_experimental",
        "binary_expected": False,
        "continuous_expected": True,
    },
    {
        "dataset": "NCATS",
        "search_tokens": "ncats;panc1;panc_1",
        "study_role": "single-cell-line PANC-1 cross-assay benchmark",
        "independence_class": "cross_assay_external_experimental",
        "binary_expected": True,
        "continuous_expected": True,
    },
    {
        "dataset": "Hoefer",
        "search_tokens": "hoefer;hofer",
        "study_role": "targeted gemcitabine-anchored multi-line PDAC screen",
        "independence_class": "cross_assay_external_experimental",
        "binary_expected": True,
        "continuous_expected": True,
    },
    {
        "dataset": "Vis",
        "search_tokens": "vis_bashi;vis",
        "study_role": "same-platform experimental replication / transfer",
        "independence_class": "same_platform_experimental_replication",
        "binary_expected": False,
        "continuous_expected": True,
    },
    {
        "dataset": "Bashi",
        "search_tokens": "vis_bashi;bashi",
        "study_role": "full-matrix experimental replication / transfer",
        "independence_class": "related_ecosystem_experimental",
        "binary_expected": True,
        "continuous_expected": True,
    },
])

display(SOURCE_REGISTRY)
SOURCE_REGISTRY.to_csv(OUT / "external_source_registry.csv", index=False)


  dataset       search_tokens                                         study_role                      independence_class  binary_expected  continuous_expected
0    Yale                yale        multi-cell-line cross-assay external screen       cross_assay_external_experimental            False                 True
1   NCATS  ncats;panc1;panc_1      single-cell-line PANC-1 cross-assay benchmark       cross_assay_external_experimental             True                 True
2  Hoefer        hoefer;hofer  targeted gemcitabine-anchored multi-line PDAC ...       cross_assay_external_experimental             True                 True
3     Vis       vis_bashi;vis  same-platform experimental replication / transfer  same_platform_experimental_replication            False                 True
4   Bashi     vis_bashi;bashi    full-matrix experimental replication / transfer          related_ecosystem_experimental             True                 True

In [8]:
SUPPORTED_EXT = {".csv", ".tsv", ".txt", ".pkl", ".pickle", ".parquet"}

def read_table(path, nrows=None):
    suf = path.suffix.lower()

    if suf == ".csv":
        return pd.read_csv(path, nrows=nrows)

    if suf in {".tsv", ".txt"}:
        return pd.read_csv(path, sep="\t", nrows=nrows)

    if suf in {".pkl", ".pickle"}:
        d = pd.read_pickle(path)
        return d.head(nrows) if nrows else d

    if suf == ".parquet":
        d = pd.read_parquet(path)
        return d.head(nrows) if nrows else d

    raise ValueError(path)


def norm_col(c):
    return re.sub(r"[^a-z0-9]+", "", str(c).lower())


def find_pair_identity_columns(df):
    """
    Return whether a table contains enough information to identify
    individual drug pairs.

    Accept either:
      1) an explicit pair-key column, or
      2) two drug-name columns.
    """
    lookup = {norm_col(c): c for c in df.columns}

    pair_candidates = {
        "pairkey",
        "drugpair",
        "drugpairgroup",
        "canonicalpair",
        "pair",
    }

    pair_col = next(
        (lookup[k] for k in pair_candidates if k in lookup),
        None,
    )

    a_candidates = {
        "drugacanon",
        "druga",
        "drug1",
        "drug1name",
        "anchor",
        "anchorname",
        "compound1",
        "compound1name",
    }

    b_candidates = {
        "drugbcanon",
        "drugb",
        "drug2",
        "drug2name",
        "library",
        "libraryname",
        "compound2",
        "compound2name",
    }

    a_col = next(
        (lookup[k] for k in a_candidates if k in lookup),
        None,
    )

    b_col = next(
        (lookup[k] for k in b_candidates if k in lookup),
        None,
    )

    usable = (
        pair_col is not None
        or (
            a_col is not None
            and b_col is not None
        )
    )

    return usable, pair_col, a_col, b_col


def find_external_outcome_columns(df):
    """
    Require at least one *experimental outcome* column.

    Model-only score/rank/metric tables are deliberately rejected.
    """
    cols = {norm_col(c): c for c in df.columns}

    exact_outcomes = {
        # Generic harmonized outputs
        "pairanypositive",
        "externalbinary",
        "externalbinarypairany",
        "binarypositivefraction",
        "externalbinarypositivefraction",
        "positivefraction",
        "meancontinuousresponse",
        "meansumaucdelta",
        "meansumaucdeltamean",
        "externalcontinuoushigherisbetter",
        "externalcontinuousmedianhigherisbetter",

        # Yale / Bliss-like outcomes
        "observedblissdiff",
        "meanobservedblissdiff",
        "meanobservedblissdiffmean",
        "maxobservedblissdiffmean",
        "blissdiff",
        "meanblissdiff",
        "bliss",
        "meanbliss",

        # NCATS/PANC-1
        "gamma",
        "meangamma",
        "label",
        "synergylabel",

        # Bashi
        "hsamatrix",
        "meanhsamatrix",
        "comboemax",
        "synergy",

        # Hoefer
        "aucdeltasum",
        "meanaucdeltasum",
        "publishedsynergy",
        "publishedsynergyfraction",

        # Vis
        "synergydeltaemax",
        "deltaemax",
        "meandeltaemax",
    }

    found = [
        original
        for key, original in cols.items()
        if key in exact_outcomes
    ]

    # Also allow clear experimental-response names not listed above.
    for c in df.columns:
        nc = norm_col(c)

        if any(
            token in nc
            for token in [
                "observedbliss",
                "hsamatrix",
                "comboemax",
                "synergydeltaemax",
                "aucdelta",
                "externalcontinuous",
                "externalbinary",
            ]
        ):
            found.append(c)

    found = list(dict.fromkeys(found))
    return len(found) > 0, found


In [9]:
selected={dataset:[source_paths[dataset]] for dataset in SOURCE_REGISTRY.dataset}
assert set(selected)==set(EXPECTED_SOURCE_SIZES)
prov_rows=[]
for dataset,paths in selected.items():
    if len(paths)!=1:raise AssertionError(f'{dataset}: expected one experimental source')
    p=paths[0]
    d=read_table(p)
    pair_ok,pair_col,ac,bc=find_pair_identity_columns(d)
    outcome_ok,out_cols=find_external_outcome_columns(d)
    if not pair_ok or not outcome_ok:
        raise ValueError(f'{dataset}: invalid input identity or endpoint schema: {rel(p)}')
    prov_rows.append({'dataset':dataset,'source_order':0,'source_role':'primary',
        'selected_input':rel(p),'file_size_bytes':p.stat().st_size,
        'sha256':sha256_file(p),'pair_col':pair_col,'drug_a_col':ac,
        'drug_b_col':bc,'experimental_outcome_cols':json.dumps(out_cols)})
provenance=pd.DataFrame(prov_rows)
provenance.to_csv(OUT/'external_input_provenance_sha256.csv',index=False)
SOURCE_REGISTRY.to_csv(OUT/'external_source_registry.csv',index=False)
display(provenance)


  dataset  source_order source_role                                     selected_input  file_size_bytes                                             sha256  pair_col    drug_a_col    drug_b_col  \
0    Yale             0     primary  results/tables/ml_comprehensive/external_valid...            96540  9f6c9ca32571b4887d1c02d25f3fd782d03a0324f6587b...  pair_key        drug_a        drug_b   
1   NCATS             0     primary  results/tables/ml_comprehensive/external_exper...            17443  f8fb0bdb3a6985a784b4a4e071467cc8e5db3adfb77de7...      None        drug_a        drug_b   
2  Hoefer             0     primary  results/tables/ml_comprehensive/external_hoefe...            19432  a5d21d81adff06ee3ce625864c4facc130f25f568481fc...  pair_key        drug_a        drug_b   
3     Vis             0     primary  results/tables/ml_comprehensive/vis_bashi_inde...            25742  e075cb536e996a9255c1c2b4f8b168cd0b0841348d450f...  pair_key  drug_a_canon  drug_b_canon   
4   Bashi           

In [10]:
def pick_col(df, candidates, contains=None):

    lookup = {
        norm_col(c): c
        for c in df.columns
    }

    for cand in candidates:

        k = norm_col(cand)

        if k in lookup:
            return lookup[k]

    if contains:

        for c in df.columns:

            nc = norm_col(c)

            if all(
                norm_col(x) in nc
                for x in contains
            ):
                return c

    return None


def split_pair_key(pk):

    if pd.isna(pk):
        return (None, None)

    s = str(pk)

    for sep in [
        "||",
        " + ",
        "+",
        "__",
        "|",
    ]:

        if sep in s:

            parts = [
                x.strip()
                for x in s.split(sep)
                if x.strip()
            ]

            if len(parts) == 2:
                return parts[0], parts[1]

    return (None, None)


# ------------------------------------------------------------
# E. Standardize one source table
# ------------------------------------------------------------

def standardize_outcome_table(
    df,
    dataset,
    source_input_path=None,
):

    d = df.copy()


    # --------------------------------------------------------
    # Restrict shared Vis/Bashi table to the requested source
    # --------------------------------------------------------

    ds_col = pick_col(
        d,
        ["dataset", "source", "study"],
    )

    if (
        ds_col is not None
        and dataset in {"Vis", "Bashi"}
    ):

        mask = (
            d[ds_col]
            .astype(str)
            .str.lower()
            .str.contains(
                dataset.lower(),
                na=False,
            )
        )

        if mask.any():
            d = d.loc[mask].copy()


    # --------------------------------------------------------
    # Pair identity
    # --------------------------------------------------------

    pair_col = pick_col(
        d,
        [
            "pair_key",
            "drug_pair_group",
            "drug_pair",
            "pair",
        ],
    )

    a_col = pick_col(
        d,
        [
            "drug_a_canon",
            "drug_a",
            "drug1",
            "drug_1",
            "anchor",
            "anchor_name",
        ],
    )

    b_col = pick_col(
        d,
        [
            "drug_b_canon",
            "drug_b",
            "drug2",
            "drug_2",
            "library",
            "library_name",
        ],
    )

    subset_col = pick_col(
        d,
        [
            "external_subset",
            "novelty_subset",
            "subset",
        ],
    )

    cell_col = pick_col(
        d,
        [
            "cell_line",
            "cellline",
            "modelid",
            "model_id",
        ],
    )


    if (
        pair_col is None
        and (
            a_col is None
            or b_col is None
        )
    ):
        raise ValueError(
            f"{dataset}: selected table has no usable "
            "pair or drug columns."
        )


    if pair_col is not None:

        parsed = d[pair_col].apply(
            split_pair_key
        )

        parsed_a = parsed.map(
            lambda x: x[0]
        )

        parsed_b = parsed.map(
            lambda x: x[1]
        )

    else:

        parsed_a = pd.Series(
            [None] * len(d),
            index=d.index,
        )

        parsed_b = pd.Series(
            [None] * len(d),
            index=d.index,
        )


    raw_a = (
        d[a_col]
        if a_col is not None
        else parsed_a
    )

    raw_b = (
        d[b_col]
        if b_col is not None
        else parsed_b
    )


    d["_drug_a"] = raw_a.map(
        canon_drug
    )

    d["_drug_b"] = raw_b.map(
        canon_drug
    )

    d["_pair_key"] = [
        make_pair_key(a, b)
        for a, b in zip(
            d["_drug_a"],
            d["_drug_b"],
        )
    ]


    if d["_pair_key"].isna().all():

        raise ValueError(
            f"{dataset}: no canonical pair keys "
            "could be constructed."
        )


    d["_source_subset"] = (
        d[subset_col].astype(str)
        if subset_col is not None
        else "unspecified"
    )


    # --------------------------------------------------------
    # Binary outcomes
    # --------------------------------------------------------

    binary_col = pick_col(
        d,
        [
            "pair_any_positive",
            "external_binary",
            "any_positive",
            "synergy_label",
            "binary_label",
            "label",
            "Synergy",
            "published_synergy",
        ],
    )


    frac_col = pick_col(
        d,
        [
            "binary_positive_fraction",
            "positive_fraction",
            "synergy_fraction",
            "published_synergy_fraction",
            "positive_cell_fraction",
        ],
    )


    # --------------------------------------------------------
    # Continuous endpoint
    # --------------------------------------------------------

    cont_col = pick_col(
        d,
        ["mean_continuous_response"],
    )

    transform = None


    if cont_col is not None:

        d["_continuous_hib"] = (
            pd.to_numeric(
                d[cont_col],
                errors="coerce",
            )
        )

        transform = (
            f"used {cont_col} as pre-standardized "
            "higher-is-better"
        )

    else:

        gamma_col = pick_col(
            d,
            [
                "Gamma",
                "mean_gamma",
                "gamma_mean",
            ],
        )


        bliss_col = pick_col(
            d,
            [
                "mean_bliss_diff",
                "mean_bliss",
                "observed_bliss_diff",
                "bliss_diff",
                "mean_observed_bliss_diff",
                "mean_observed_bliss_diff_mean",
                "max_observed_bliss_diff_mean",
            ],
            contains=[
                "observed",
                "bliss",
                "diff",
            ],
        )


        hsa_col = pick_col(
            d,
            [
                "HSA_matrix",
                "mean_hsa_matrix",
                "mean_hsa",
            ],
        )


        # Höfer aliases
        aucd_col = pick_col(
            d,
            [
                "AUCΔ_sum [%]",
                "AUC∆_sum [%]",
                "AUC_delta_sum",
                "mean_auc_delta_sum",
                "mean_auc_delta",
                "mean_sum_auc_delta",
                "mean_sum_auc_delta_mean",
            ],
            contains=[
                "auc",
                "delta",
            ],
        )


        demax_col = pick_col(
            d,
            [
                "SYNERGY_DELTA_EMAX",
                "mean_delta_emax",
                "delta_emax",
                "mean_synergy_delta_emax",
            ],
        )


        if gamma_col is not None:

            d["_continuous_hib"] = (
                -pd.to_numeric(
                    d[gamma_col],
                    errors="coerce",
                )
            )

            transform = (
                f"-1 * {gamma_col} "
                "(lower native Gamma = stronger synergy)"
            )


        elif bliss_col is not None:

            d["_continuous_hib"] = (
                pd.to_numeric(
                    d[bliss_col],
                    errors="coerce",
                )
            )

            transform = (
                f"{bliss_col} "
                "(higher = stronger Bliss excess)"
            )


        elif hsa_col is not None:

            d["_continuous_hib"] = (
                pd.to_numeric(
                    d[hsa_col],
                    errors="coerce",
                )
            )

            transform = (
                f"{hsa_col} "
                "(higher = stronger HSA excess)"
            )


        elif aucd_col is not None:

            d["_continuous_hib"] = (
                pd.to_numeric(
                    d[aucd_col],
                    errors="coerce",
                )
            )

            transform = (
                f"{aucd_col} "
                "(higher = stronger response)"
            )


        elif demax_col is not None:

            d["_continuous_hib"] = (
                pd.to_numeric(
                    d[demax_col],
                    errors="coerce",
                )
            )

            transform = (
                f"{demax_col} "
                "(higher = stronger response)"
            )


        else:

            d["_continuous_hib"] = np.nan

            transform = (
                "continuous endpoint unavailable"
            )


    # --------------------------------------------------------
    # Standardize binary row labels
    # --------------------------------------------------------

    if binary_col is not None:

        raw_bin = d[binary_col]

        if raw_bin.dtype == bool:

            d["_binary_row"] = (
                raw_bin.astype(float)
            )

        else:

            s = (
                raw_bin
                .astype(str)
                .str.strip()
                .str.lower()
            )

            mapv = {
                "true": 1.0,
                "false": 0.0,
                "yes": 1.0,
                "no": 0.0,
                "synergy": 1.0,
                "synergistic": 1.0,
                "positive": 1.0,
                "negative": 0.0,
                "1": 1.0,
                "0": 0.0,
            }

            num = pd.to_numeric(
                raw_bin,
                errors="coerce",
            )

            d["_binary_row"] = (
                num.where(
                    num.notna(),
                    s.map(mapv),
                )
            )

    else:

        d["_binary_row"] = np.nan


    if frac_col is not None:

        d["_positive_fraction_input"] = (
            pd.to_numeric(
                d[frac_col],
                errors="coerce",
            )
        )

    else:

        d["_positive_fraction_input"] = np.nan


    # --------------------------------------------------------
    # Aggregate to one row per canonical drug pair
    # --------------------------------------------------------

    rows = []


    grouped = (
        d
        .dropna(
            subset=["_pair_key"]
        )
        .groupby(
            "_pair_key",
            sort=False,
        )
    )


    for pk, g in grouped:

        a, b = split_pair_key(pk)


        frac_vals = (
            g["_positive_fraction_input"]
            .dropna()
        )

        bin_vals = (
            g["_binary_row"]
            .dropna()
        )


        if len(frac_vals):

            positive_fraction = float(
                frac_vals.mean()
            )

            pair_any = float(
                positive_fraction > 0
            )

        elif len(bin_vals):

            positive_fraction = float(
                bin_vals.mean()
            )

            pair_any = float(
                bin_vals.max() > 0
            )

        else:

            positive_fraction = np.nan
            pair_any = np.nan


        cont_vals = (
            g["_continuous_hib"]
            .dropna()
        )

        cont_mean = (
            float(cont_vals.mean())
            if len(cont_vals)
            else np.nan
        )

        cont_median = (
            float(cont_vals.median())
            if len(cont_vals)
            else np.nan
        )


        source_subset_vals = (
            g["_source_subset"]
            .dropna()
            .astype(str)
        )

        source_subset = (
            source_subset_vals
            .mode()
            .iloc[0]
            if len(source_subset_vals)
            else "unspecified"
        )


        n_cell = (
            g[cell_col].nunique(
                dropna=True
            )
            if cell_col is not None
            else np.nan
        )


        rows.append(
            {
                "dataset": dataset,
                "source_subset": source_subset,
                "pair_key": pk,
                "drug_a_canon": a,
                "drug_b_canon": b,
                "n_source_rows": len(g),
                "n_external_cell_lines": n_cell,
                "external_binary_pair_any": pair_any,
                "external_binary_positive_fraction": (
                    positive_fraction
                ),
                "external_continuous_higher_is_better": (
                    cont_mean
                ),
                "external_continuous_median_higher_is_better": (
                    cont_median
                ),
                "continuous_transform": transform,
                "source_input_path": (
                    rel(source_input_path)
                    if source_input_path is not None
                    else ""
                ),
            }
        )


    out = pd.DataFrame(rows)


    if out["pair_key"].duplicated().any():

        raise ValueError(
            f"{dataset}: duplicate pair keys remained "
            "after aggregation."
        )


    return out


# ------------------------------------------------------------
# F. Harmonize the five sources
# ------------------------------------------------------------

harmonized_parts = []


for dataset, paths in selected.items():

    print(
        "\n"
        + "=" * 70
    )

    print(
        dataset,
        "selected source table(s):",
    )


    # Final notebook uses exactly ONE genuine experimental
    # pair-level source per dataset.
    if len(paths) != 1:

        raise AssertionError(
            f"{dataset}: expected exactly one final "
            f"experimental source table, got {len(paths)}."
        )


    p = paths[0]

    raw = read_table(p)


    print(
        " ",
        p,
    )

    print(
        "    Raw shape:",
        raw.shape,
    )


    h = standardize_outcome_table(
        raw,
        dataset,
        source_input_path=p,
    )


    print(
        "    Harmonized pairs:",
        len(h),
    )

    print(
        "    Binary nonmissing:",
        int(
            h[
                "external_binary_pair_any"
            ]
            .notna()
            .sum()
        ),
    )

    print(
        "    Continuous nonmissing:",
        int(
            h[
                "external_continuous_higher_is_better"
            ]
            .notna()
            .sum()
        ),
    )


    harmonized_parts.append(h)


external_units = pd.concat(
    harmonized_parts,
    ignore_index=True,
)


# ------------------------------------------------------------
# G. Source-level endpoint summary
# ------------------------------------------------------------

source_endpoint_summary = (
    external_units
    .groupby("dataset")
    .agg(
        n_pairs=(
            "pair_key",
            "nunique",
        ),
        n_binary=(
            "external_binary_pair_any",
            lambda x: x.notna().sum(),
        ),
        n_continuous=(
            "external_continuous_higher_is_better",
            lambda x: x.notna().sum(),
        ),
        n_binary_classes=(
            "external_binary_pair_any",
            lambda x: x.dropna().nunique(),
        ),
    )
    .reset_index()
)


display(
    source_endpoint_summary
)


# ------------------------------------------------------------
# H. FINAL endpoint-contract audit
# ------------------------------------------------------------

contract_rows = []


for _, reg in SOURCE_REGISTRY.iterrows():

    dataset = reg["dataset"]

    s = source_endpoint_summary.loc[
        source_endpoint_summary[
            "dataset"
        ].eq(dataset)
    ]


    if s.empty:

        raise AssertionError(
            f"{dataset}: missing from harmonized "
            "external units."
        )


    s = s.iloc[0]


    binary_expected = bool(
        reg["binary_expected"]
    )

    continuous_expected = bool(
        reg["continuous_expected"]
    )


    if binary_expected:

        binary_ok = (
            int(s["n_binary"]) > 0
        )

        binary_two_classes_ok = (
            int(
                s["n_binary_classes"]
            )
            >= 2
        )

    else:

        binary_ok = True
        binary_two_classes_ok = True


    if continuous_expected:

        continuous_ok = (
            int(
                s["n_continuous"]
            )
            > 0
        )

    else:

        continuous_ok = True


    contract_ok = (
        binary_ok
        and binary_two_classes_ok
        and continuous_ok
    )


    contract_rows.append(
        {
            "dataset": dataset,
            "binary_expected": binary_expected,
            "continuous_expected": continuous_expected,
            "n_pairs": int(
                s["n_pairs"]
            ),
            "n_binary": int(
                s["n_binary"]
            ),
            "n_binary_classes": int(
                s["n_binary_classes"]
            ),
            "n_continuous": int(
                s["n_continuous"]
            ),
            "binary_present_ok": binary_ok,
            "binary_two_classes_ok": (
                binary_two_classes_ok
            ),
            "continuous_present_ok": (
                continuous_ok
            ),
            "contract_ok": contract_ok,
        }
    )


endpoint_contract_audit = (
    pd.DataFrame(
        contract_rows
    )
)


display(
    endpoint_contract_audit
)


endpoint_contract_audit.to_csv(
    OUT
    / "external_endpoint_contract_audit.csv",
    index=False,
)


if not endpoint_contract_audit[
    "contract_ok"
].all():

    bad = endpoint_contract_audit.loc[
        ~endpoint_contract_audit[
            "contract_ok"
        ]
    ]

    raise AssertionError(
        "External endpoint contract failed.\n"
        + bad.to_string(
            index=False
        )
    )


# ------------------------------------------------------------
# I. Critical contamination guards
# ------------------------------------------------------------

all_selected_paths = [
    str(p).lower()
    for paths in selected.values()
    for p in paths
]


for p in all_selected_paths:

    if (
        "audit_internal_training" in p
        or "internal_training_table_preview" in p
    ):

        raise AssertionError(
            "INTERNAL TRAINING AUDIT TABLE ENTERED "
            f"EXTERNAL BENCHMARK: {p}"
        )


# Yale must remain the real 820-pair external Bliss set.
yale_final = external_units.loc[
    external_units[
        "dataset"
    ].eq("Yale")
]


if len(yale_final) != 820:

    raise AssertionError(
        "Unexpected Yale pair count after correction: "
        f"{len(yale_final)}; expected 820."
    )


if (
    yale_final[
        "external_continuous_higher_is_better"
    ]
    .notna()
    .sum()
    != 820
):

    raise AssertionError(
        "Yale Bliss endpoint is incomplete."
    )


if (
    yale_final[
        "external_binary_pair_any"
    ]
    .notna()
    .any()
):

    raise AssertionError(
        "Yale unexpectedly contains a binary endpoint. "
        "Check that no internal-training table entered "
        "the benchmark."
    )


print()
print(
    "EXTERNAL ENDPOINT CONTRACTS PASSED."
)

print(
    "Yale internal-training contamination guard PASSED."
)

print(
    "Correct final source selection PASSED."
)



Yale selected source table(s):
  C:\Users\Käyttäjä\Documents\PDAC-drug-combination-ML-MSc-thesis\results\tables\ml_comprehensive\external_validation_yale_pdac\yale_pair_level_external_candidate_summary.csv
    Raw shape: (820, 9)
    Harmonized pairs: 820
    Binary nonmissing: 0
    Continuous nonmissing: 820

NCATS selected source table(s):
  C:\Users\Käyttäjä\Documents\PDAC-drug-combination-ML-MSc-thesis\results\tables\ml_comprehensive\external_experimental_validation\ncats_from_raw_training_set.csv
    Raw shape: (496, 4)
    Harmonized pairs: 496
    Binary nonmissing: 496
    Continuous nonmissing: 496

Hoefer selected source table(s):
  C:\Users\Käyttäjä\Documents\PDAC-drug-combination-ML-MSc-thesis\results\tables\ml_comprehensive\external_hoefer_2025_cross_assay_validation\hoefer_inhibitor_pair_level_summary.csv
    Raw shape: (146, 12)
    Harmonized pairs: 146
    Binary nonmissing: 146
    Continuous nonmissing: 146

Vis selected source table(s):
  C:\Users\Käyttäjä\Documen

  dataset  n_pairs  n_binary  n_continuous  n_binary_classes
0   Bashi      109       109           109                 2
1  Hoefer      146       146           146                 2
2   NCATS      496       496           496                 2
3     Vis       48         0            48                 0
4    Yale      820         0           820                 0

  dataset  binary_expected  continuous_expected  n_pairs  n_binary  n_binary_classes  n_continuous  binary_present_ok  binary_two_classes_ok  continuous_present_ok  contract_ok
0    Yale            False                 True      820         0                 0           820               True                   True                   True         True
1   NCATS             True                 True      496       496                 2           496               True                   True                   True         True
2  Hoefer             True                 True      146       146                 2           146               True                   True                   True         True
3     Vis            False                 True       48         0                 0            48               True                   True                   True         True
4   Bashi             True                 True      109       109                 2           109               Tr


EXTERNAL ENDPOINT CONTRACTS PASSED.
Yale internal-training contamination guard PASSED.
Correct final source selection PASSED.


## 6. Recomputing drug/pair novelty against the exact internal training library

Same-pair, one-cold and both-cold categories are inferred solely from internal training drug identities and drug pairs. `known_drugs_unseen_pair` is retained in the original definition (though no such pairs are expected in this fully crossed 26-drug library).

In [11]:
def novelty(a, b, pair_key):
    ak = a in internal_drugs
    bk = b in internal_drugs

    if pair_key in internal_pairs:
        return "same_pair_reproducibility"
    if ak and bk:
        return "known_drugs_unseen_pair"
    if ak ^ bk:
        return "one_cold_drug"
    return "both_cold_drugs"

external_units["canonical_external_subset"] = [
    novelty(a, b, p)
    for a, b, p in zip(
        external_units["drug_a_canon"],
        external_units["drug_b_canon"],
        external_units["pair_key"],
    )
]

external_units = external_units.merge(
    SOURCE_REGISTRY[
        ["dataset", "study_role", "independence_class"]
    ],
    on="dataset",
    how="left",
)

display(
    external_units.groupby(
        ["dataset", "canonical_external_subset"]
    ).agg(
        n_pairs=("pair_key", "nunique"),
        binary_nonmissing=("external_binary_pair_any", lambda x: x.notna().sum()),
        continuous_nonmissing=("external_continuous_higher_is_better", lambda x: x.notna().sum()),
    ).reset_index()
)


   dataset  canonical_external_subset  n_pairs  binary_nonmissing  continuous_nonmissing
0    Bashi            both_cold_drugs       58                 58                     58
1    Bashi              one_cold_drug       43                 43                     43
2    Bashi  same_pair_reproducibility        8                  8                      8
3   Hoefer              one_cold_drug      133                133                    133
4   Hoefer  same_pair_reproducibility       13                 13                     13
5    NCATS            both_cold_drugs      465                465                    465
6    NCATS              one_cold_drug       31                 31                     31
7      Vis              one_cold_drug       16                  0                     16
8      Vis  same_pair_reproducibility       32                  0                     32
9     Yale            both_cold_drugs      465                  0                    465
10    Yale           

## 7. Molecular-structure sources and chemical-form quality control
Molecular structures for external compounds are collected from explicitly defined source files, the original NCATS experimental structures and a locally stored structure cache. Drug names are harmonised before selecting molecular structures, and source priority is fixed.

For drugs represented in the internal training data, the stored training fingerprints are used. External-only compounds are assigned 512-bit Morgan fingerprints generated from the selected source SMILES. Alternative source structures are recorded and compared to identify differences affecting the resulting fingerprints.

Where source representations produce different fingerprints, RDKit charge-parent and canonical-tautomer standardisation is used as a chemical-form equivalence diagnostic. This diagnostic does not alter the primary modelling fingerprints.

The selected structure source, alternative representations, chemical-form audit and source-file hashes are exported to support reproducibility.

Two previously unresolved external compounds were assigned literature-backed molecular structures. GSK650394A was identified from the original Langdon et al. screening study and independently documented chemical identifiers. AZ3202 was identified as compound 15 in Tao et al. (2014), using the published systematic chemical name and supporting analytical characterisation. Their selected SMILES representations are stored in `external_approved_smiles.csv` and validated against expected molecular formulas and InChIKeys before prediction. These additions improve fingerprint coverage without modifying the internal training data or model specifications.

In [13]:
# Source order is explicit. The first source has priority, but any conflicting
# structures for evaluated cold drugs are reported and checked before use.
STRUCTURE_SOURCES = [
    ('data/processed/external_validation/external_approved_smiles.csv','drug_canon','smiles'),
    ('data/raw/external_validation/hoefer_2025/hoefer_inhibitor_smiles_manual_overrides.csv','inhibitor','smiles'),
    ('data/raw/external_validation/hoefer_2025/hoefer_inhibitor_smiles_cache.csv','inhibitor','canonical_smiles'),
    ('data/processed/dataset_assembly/drug_smiles_master_corrected.csv','drug_norm','corrected_smiles'),
    ('data/processed/dataset_assembly/drug_master_enriched_with_drugcomb_identifiers.csv','drug_name','smiles'),
    ('data/processed/dataset_assembly/drug_master_jaaks_with_drugcomb_identifiers.csv','drug_name','smiles'),
    ('data/processed/dataset_assembly/drug_master.csv','drug_name','smiles'),
    ('data/raw/DepMap_v2/PortalCompounds.csv','CompoundName','SMILES'),
]

def canonical_smiles(s):
    if pd.isna(s) or not str(s).strip():return None
    mol=Chem.MolFromSmiles(str(s).strip())
    if mol is None:return None
    return Chem.MolToSmiles(mol,isomericSmiles=True)

external_drugs=sorted(set(external_units.drug_a_canon.dropna())|
                      set(external_units.drug_b_canon.dropna()))
external_drug_set=set(external_drugs)
structure_candidates={}
structure_provenance=[]

def offer_structure(name,smiles,source):
    d=canon_drug(name)
    if not d or d not in external_drug_set:return
    s=canonical_smiles(smiles)
    if s is None:return
    structure_candidates.setdefault(d,[]).append((s,source))

for relpath,name_col,smiles_col in STRUCTURE_SOURCES:
    path=repo_path(relpath)
    if not path.is_file():
        raise FileNotFoundError(
            "Required molecular-structure source is missing: "
            f"{relpath}"
            "Install the versioned source inputs before running external validation."
        )

    d=pd.read_csv(path,low_memory=False)
    if name_col not in d or smiles_col not in d:
        raise KeyError(f'Structure source {relpath}: missing {name_col!r} or {smiles_col!r}')
    structure_provenance.append({'source':rel(path),'sha256':sha256_file(path),'rows':len(d)})
    for n,s in d[[name_col,smiles_col]].itertuples(index=False,name=None):
        offer_structure(n,s,'local:'+rel(path))

# The original NCATS input carries structures, despite not having 'SMILES1'/'SMILES2'.
for name_col,smi_col in [('Sample Name 1','Structure 1'),('Sample Name 2','Structure 2')]:
    for n,s in ncats_raw[[name_col,smi_col]].itertuples(index=False,name=None):
        offer_structure(n,s,'ncats_raw:'+rel(NCATS_INPUT))
structure_provenance.append({'source':rel(NCATS_INPUT),'sha256':sha256_file(NCATS_INPUT),
                             'rows':len(ncats_raw)})

# A *source-backed*, separately versionable cache (not any old model-output file).
if CACHE_PATH.is_file():
    cached=pd.read_csv(CACHE_PATH,low_memory=False)
    required_cache={'drug_canon','smiles'}
    if not required_cache.issubset(cached):raise KeyError(f'PubChem cache missing {required_cache-set(cached.columns)}')
    for row in cached.itertuples(index=False):
        offer_structure(row.drug_canon,row.smiles,'pubchem_cache:'+rel(CACHE_PATH))
    structure_provenance.append({'source':rel(CACHE_PATH),'sha256':sha256_file(CACHE_PATH),'rows':len(cached)})
else:
    cached=pd.DataFrame(columns=['drug_canon','smiles','source'])


# Select the first valid structure according to the
# explicitly defined STRUCTURE_SOURCES priority.
# Preserve the selected source for provenance auditing.

structure_bank = {}
structure_source = {}

for drug, choices in sorted(structure_candidates.items()):
    if not choices:
        continue

    selected_smiles, selected_source = choices[0]

    structure_bank[drug] = selected_smiles
    structure_source[drug] = selected_source

assert set(structure_bank) == set(structure_source)

print(
    "Compounds with selected molecular structures:",
    len(structure_bank)
)


# Verify the two literature-backed recovered structures.
# These checks do not modify the selected fingerprints.

from rdkit.Chem import rdMolDescriptors

APPROVED_SMILES_SOURCE = (
    "local:data/processed/external_validation/"
    "external_approved_smiles.csv"
)

RECOVERED_STRUCTURES = {
    "GSK650394A": {
        "formula": "C25H22N2O2",
        "inchikey": "WVSBGSNVCDAMCF-UHFFFAOYSA-N",
    },
    "AZ3202": {
        "formula": "C35H33N5O4S2",
        "inchikey": "UBVFEVVNFNKDNY-UHFFFAOYSA-N",
    },
}

for drug, expected in RECOVERED_STRUCTURES.items():
    assert drug in external_drug_set, (
        f"{drug} is absent from the external registry."
    )

    assert drug not in internal_drugs, (
        f"{drug} was expected to be an external cold drug."
    )

    assert structure_source.get(drug) == APPROVED_SMILES_SOURCE, (
        f"{drug}: the approved source was not selected. "
        f"Actual source: {structure_source.get(drug)}"
    )

    mol = Chem.MolFromSmiles(structure_bank[drug])

    assert mol is not None, f"{drug}: invalid SMILES."

    formula = rdMolDescriptors.CalcMolFormula(mol)
    inchikey = Chem.MolToInchiKey(mol)

    assert formula == expected["formula"], (
        f"{drug}: unexpected formula {formula}"
    )

    assert inchikey == expected["inchikey"], (
        f"{drug}: unexpected InChIKey {inchikey}"
    )

    print(f"{drug}: approved structure verified.")

print("Both recovered molecular structures passed verification.")


# Freeze the six currently selected, source-backed chemical
# representations. These are raw-source choices, not a
# newly standardised fingerprint representation.

STRUCTURE_SOURCE_LOCKS = {
    "GMX1778":
        "local:data/processed/dataset_assembly/"
        "drug_master_enriched_with_drugcomb_identifiers.csv",

    "IMATINIB":
        "local:data/raw/external_validation/hoefer_2025/"
        "hoefer_inhibitor_smiles_cache.csv",

    "MOTESANIB":
        "local:data/raw/external_validation/hoefer_2025/"
        "hoefer_inhibitor_smiles_cache.csv",

    "PAZOPANIB":
        "local:data/raw/external_validation/hoefer_2025/"
        "hoefer_inhibitor_smiles_cache.csv",

    "RIGOSERTIB":
        "local:data/raw/external_validation/hoefer_2025/"
        "hoefer_inhibitor_smiles_cache.csv",

    "SEPANTRONIUMBROMIDE":
        "local:data/raw/DepMap_v2/PortalCompounds.csv",
}

for drug, expected_source in STRUCTURE_SOURCE_LOCKS.items():
    matching = {
        smi for smi, src in structure_candidates[drug]
        if src == expected_source
    }

    if len(matching) != 1:
        raise ValueError(
            f"{drug}: expected exactly one structure from "
            f"{expected_source}; found {len(matching)}"
        )

    expected_smiles = next(iter(matching))

    if (
        structure_source[drug] != expected_source
        or structure_bank[drug] != expected_smiles
    ):
        raise ValueError(
            f"{drug}: selected structure changed from its "
            "audited source. Review before continuing."
        )

print("Six source-backed structure choices are locked.")


# Molecular structures are selected using a fixed, documented source priority.
# Alternative valid structures are retained for chemical provenance and fingerprint-sensitivity auditing.

from rdkit.Chem import rdFingerprintGenerator
_morgan_audit_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=512)

def _audit_morgan_bits(s):
    mol = Chem.MolFromSmiles(s)
    if mol is None:
        raise ValueError(f'Invalid SMILES survived canonicalisation: {s!r}')
    return _morgan_audit_gen.GetFingerprint(mol).ToBitString()

candidate_audit_rows = []
conflict_summary_rows = []
for drug, choices in sorted(structure_candidates.items()):
    chosen_smi, chosen_source = structure_bank[drug], structure_source[drug]
    selected_bits = _audit_morgan_bits(chosen_smi)
    fp_by_smi = {s: _audit_morgan_bits(s) for s, _ in choices}
    unique_structures = {s for s, _ in choices}
    unique_fps = set(fp_by_smi.values())
    distances = []
    for priority, (smi, src) in enumerate(choices):
        hamming = sum(a != b for a, b in zip(fp_by_smi[smi], selected_bits))
        distances.append(hamming)
        candidate_audit_rows.append({
            'drug_canon': drug,
            'candidate_priority': priority,
            'selected': priority == 0,
            'structure_source': src,
            'candidate_smiles': smi,
            'is_internal_drug_stored_fp': drug in internal_drugs,
            'morgan_512_bit_differences_from_selected': hamming,
        })
    if len(unique_structures) > 1:
        conflict_summary_rows.append({
            'drug_canon': drug,
            'n_distinct_smiles': len(unique_structures),
            'n_distinct_morgan_512': len(unique_fps),
            'max_bit_difference_from_selected': max(distances),
            'all_candidates_morgan_equivalent': len(unique_fps) == 1,
            'is_internal_drug_stored_fp': drug in internal_drugs,
            'selected_source': chosen_source,
            'selected_smiles': chosen_smi,
            'has_explicit_approved_override': any(
                src == APPROVED_SMILES_SOURCE for _, src in choices
            ),
        })

candidate_audit = pd.DataFrame(candidate_audit_rows)
conflicts = pd.DataFrame(conflict_summary_rows, columns=[
    'drug_canon','n_distinct_smiles','n_distinct_morgan_512',
    'max_bit_difference_from_selected','all_candidates_morgan_equivalent',
    'is_internal_drug_stored_fp','selected_source','selected_smiles',
    'has_explicit_approved_override'])
candidate_audit.to_csv(OUT / 'external_structure_candidate_audit.csv', index=False)
conflicts.to_csv(OUT / 'external_structure_conflicts.csv', index=False)
pd.DataFrame(structure_provenance).to_csv(
    OUT / 'external_structure_source_hashes.csv', index=False)

print('External compounds:', len(external_drugs),
      'with candidate structures:', len(structure_bank))
print('Drugs with multiple canonical SMILES:', len(conflicts))
if len(conflicts):
    display(conflicts.drop(columns=['selected_smiles']))
    print('Audit of selected source and every alternative:',
          rel(OUT / 'external_structure_candidate_audit.csv'))


# Chemical-form equivalence audit.
# This does NOT alter any fingerprint used for prediction.

from rdkit.Chem.MolStandardize import rdMolStandardize

form_sensitive_drugs = sorted(
    conflicts.loc[
        (~conflicts["is_internal_drug_stored_fp"])
        & (~conflicts["all_candidates_morgan_equivalent"]),
        "drug_canon"
    ].tolist()
)

tautomer_enumerator = rdMolStandardize.TautomerEnumerator()

form_audit_rows = []

for drug in form_sensitive_drugs:

    unique_candidates = sorted(
        set(structure_candidates[drug])
    )

    parent_representations = set()

    for smiles, source in unique_candidates:

        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            raise ValueError(f"Invalid SMILES for {drug}")

        # Used ONLY for comparing candidate chemical forms.
        charge_parent = rdMolStandardize.ChargeParent(mol)

        tautomer_parent = tautomer_enumerator.Canonicalize(
            charge_parent
        )

        parent_smiles = Chem.MolToSmiles(
            tautomer_parent,
            isomericSmiles=True
        )

        parent_representations.add(parent_smiles)

        form_audit_rows.append({
            "drug_canon": drug,
            "source": source,
            "source_smiles": smiles,
            "standardized_parent_smiles": parent_smiles,
            "selected_for_prediction": (
                smiles == structure_bank[drug]
                and source == structure_source[drug]
            ),
            "rdkit_version": rdBase.rdkitVersion,
        })

    if len(parent_representations) != 1:
        raise ValueError(
            f"{drug}: candidate structures do not converge "
            "to the same standardized parent/tautomer. "
            "Manual chemical-identity review is required."
        )

form_equivalence_audit = pd.DataFrame(
    form_audit_rows,
    columns=[
        "drug_canon",
        "source",
        "source_smiles",
        "standardized_parent_smiles",
        "selected_for_prediction",
        "rdkit_version",
    ]
)

form_equivalence_audit.to_csv(
    OUT / "external_structure_form_equivalence_audit.csv",
    index=False
)

print("Cold drugs with different raw Morgan fingerprints:",
      len(form_sensitive_drugs))

print("Drugs passing standardized-form equivalence:",
      form_equivalence_audit["drug_canon"].nunique())

print(
    "All audited alternative forms share the same "
    "standardized parent/tautomer representation."
)

print(
    "Original source-backed fingerprints remain unchanged."
)

[10:33:14] SMILES Parse Error: syntax error while parsing: CN1CCN(CC1)C2=CC3=C(C=C2)NC(=C4C(=C5C(=NC4=O)C=CC=C5F)N)N3;
[10:33:14] SMILES Parse Error: check for mistakes around position 59:
[10:33:14] =NC4=O)C=CC=C5F)N)N3;
[10:33:14] ~~~~~~~~~~~~~~~~~~~~^
[10:33:14] SMILES Parse Error: Failed parsing SMILES 'CN1CCN(CC1)C2=CC3=C(C=C2)NC(=C4C(=C5C(=NC4=O)C=CC=C5F)N)N3;' for input: 'CN1CCN(CC1)C2=CC3=C(C=C2)NC(=C4C(=C5C(=NC4=O)C=CC=C5F)N)N3;'
[10:33:15] SMILES Parse Error: syntax error while parsing: CN1CCN(CC1)CC(=O)N(C)C2=CC=C(C=C2)N/C(=C3/C4=C(C=C(C=C4)C(=O)OC)NC3=O)/C5=CC=CC=C5;
[10:33:15] SMILES Parse Error: check for mistakes around position 83:
[10:33:15] C)NC3=O)/C5=CC=CC=C5;
[10:33:15] ~~~~~~~~~~~~~~~~~~~~^
[10:33:15] SMILES Parse Error: Failed parsing SMILES 'CN1CCN(CC1)CC(=O)N(C)C2=CC=C(C=C2)N/C(=C3/C4=C(C=C(C=C4)C(=O)OC)NC3=O)/C5=CC=CC=C5;' for input: 'CN1CCN(CC1)CC(=O)N(C)C2=CC=C(C=C2)N/C(=C3/C4=C(C=C(C=C4)C(=O)OC)NC3=O)/C5=CC=CC=C5;'
[10:33:15] SMILES Parse Error: syntax erro

Compounds with selected molecular structures: 232
GSK650394A: approved structure verified.
AZ3202: approved structure verified.
Both recovered molecular structures passed verification.
Six source-backed structure choices are locked.
External compounds: 235 with candidate structures: 232
Drugs with multiple canonical SMILES: 17


             drug_canon  n_distinct_smiles  n_distinct_morgan_512  max_bit_difference_from_selected  all_candidates_morgan_equivalent  is_internal_drug_stored_fp                                    selected_source  \
0                BI2536                  2                      1                                 0                              True                       False  local:data/processed/dataset_assembly/drug_mas...   
1            BORTEZOMIB                  2                      1                                 0                              True                       False  local:data/processed/dataset_assembly/drug_mas...   
2           CARFILZOMIB                  2                      1                                 0                              True                       False  local:data/processed/dataset_assembly/drug_mas...   
3             CISPLATIN                  2                      2                                 8                             False   

Audit of selected source and every alternative: results/tables/ml_comprehensive/external_experimental_validation/external_structure_candidate_audit.csv
Cold drugs with different raw Morgan fingerprints: 6
Drugs passing standardized-form equivalence: 6
All audited alternative forms share the same standardized parent/tautomer representation.
Original source-backed fingerprints remain unchanged.


[10:33:18] Initializing MetalDisconnector
[10:33:18] Running MetalDisconnector
[10:33:18] Initializing Normalizer
[10:33:18] Running Normalizer
[10:33:18] Running LargestFragmentChooser
[10:33:18] Running Uncharger
[10:33:18] Initializing MetalDisconnector
[10:33:18] Running MetalDisconnector
[10:33:18] Initializing Normalizer
[10:33:18] Running Normalizer
[10:33:18] Initializing MetalDisconnector
[10:33:18] Running MetalDisconnector
[10:33:18] Initializing Normalizer
[10:33:18] Running Normalizer
[10:33:18] Running LargestFragmentChooser
[10:33:18] Running Uncharger
[10:33:18] Initializing MetalDisconnector
[10:33:18] Running MetalDisconnector
[10:33:18] Initializing Normalizer
[10:33:18] Running Normalizer
[10:33:18] Initializing MetalDisconnector
[10:33:18] Running MetalDisconnector
[10:33:18] Initializing Normalizer
[10:33:18] Running Normalizer
[10:33:18] Running LargestFragmentChooser
[10:33:18] Fragment: CS(=O)(=O)O
[10:33:18] New largest fragment: CS(=O)(=O)O (9)
[10:33:18] Fra

## 8. Internal Morgan fingerprints and compatibility validation

In [15]:
internal_fp_lookup = {}

tmp_a = train[[PA] + drug_a_fp].dropna(subset=[PA]).drop_duplicates(subset=[PA])
for _, row in tmp_a.iterrows():
    drug = canon_drug(row[PA])
    arr = pd.to_numeric(row[drug_a_fp], errors="coerce").to_numpy(np.float32)
    if len(arr) != 512 or np.isnan(arr).any():
        raise ValueError(f"Invalid stored fingerprint for {drug} in A role.")
    internal_fp_lookup[drug] = arr

tmp_b = train[[PB] + drug_b_fp].dropna(subset=[PB]).drop_duplicates(subset=[PB])
for _, row in tmp_b.iterrows():
    drug = canon_drug(row[PB])
    arr = pd.to_numeric(row[drug_b_fp], errors="coerce").to_numpy(np.float32)
    if len(arr) != 512 or np.isnan(arr).any():
        raise ValueError(f"Invalid stored fingerprint for {drug} in B role.")
    if drug in internal_fp_lookup and not np.array_equal(internal_fp_lookup[drug], arr):
        raise ValueError(f"Inconsistent stored fingerprint for internal drug {drug}.")
    internal_fp_lookup[drug] = arr

print("Stored internal fingerprints:", len(internal_fp_lookup))


Stored internal fingerprints: 26


In [16]:

# Check whether source-derived Morgan fingerprints agree
# with the stored internal training fingerprints.
#
# Diagnostic only: does not modify training or scoring.

from rdkit.Chem import rdFingerprintGenerator

compat_gen = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=512,
)

def audit_fp_from_smiles(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        raise ValueError(f"Invalid audited SMILES: {smiles}")

    fp = compat_gen.GetFingerprint(mol)
    arr = np.zeros(512, dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

# Check that a given internal drug is not represented by
# multiple inconsistent stored fingerprints.
for role, name_col, bit_cols in [
    ("A", PA, drug_a_fp),
    ("B", PB, drug_b_fp),
]:
    for drug, group in train.groupby(name_col):
        values = group[bit_cols].to_numpy(dtype=float)

        if not np.all(values == values[0]):
            raise ValueError(
                f"Stored fingerprints vary within role "
                f"{role} for drug {drug}"
            )

rows = []

for drug, stored_fp in sorted(internal_fp_lookup.items()):
    stored = np.asarray(stored_fp, dtype=np.int8)

    choices = {
        smiles
        for smiles, source in structure_candidates.get(drug, [])
    }

    distances = [
        int(np.count_nonzero(
            stored != audit_fp_from_smiles(smiles)
        ))
        for smiles in choices
    ]

    selected_smiles = structure_bank.get(drug)

    selected_distance = (
        int(np.count_nonzero(
            stored != audit_fp_from_smiles(selected_smiles)
        ))
        if selected_smiles is not None else np.nan
    )

    rows.append({
        "drug": drug,
        "n_candidate_structures": len(choices),
        "selected_source": structure_source.get(drug),
        "selected_bit_difference": selected_distance,
        "minimum_candidate_bit_difference":
            min(distances) if distances else np.nan,
        "any_candidate_matches_stored":
            any(d == 0 for d in distances),
    })

training_fp_compatibility = pd.DataFrame(rows)

display(training_fp_compatibility)


# Distinguish missing comparison data from an actual
# disagreement with the stored training fingerprint.

internal_not_in_external = sorted(
    set(internal_fp_lookup) - set(external_drugs)
)

comparable = training_fp_compatibility.loc[
    training_fp_compatibility["n_candidate_structures"].gt(0)
].copy()

not_comparable = training_fp_compatibility.loc[
    training_fp_compatibility["n_candidate_structures"].eq(0)
].copy()

mismatches = comparable.loc[
    ~comparable["any_candidate_matches_stored"]
]

print("Internal training drugs:", len(internal_fp_lookup))
print("Internal drugs appearing in external data:",
      len(set(internal_fp_lookup) & set(external_drugs)))
print("Internal drugs absent from external data:",
      internal_not_in_external)
print("Drugs with comparable source SMILES:", len(comparable))
print("Exact fingerprint matches:",
      int(comparable["any_candidate_matches_stored"].sum()))
print("Drugs without comparable source SMILES:",
      not_comparable["drug"].tolist())

if not mismatches.empty:
    display(mismatches)
    raise ValueError(
        "A source-derived fingerprint disagrees with "
        "the stored internal fingerprint. Investigate "
        "before finalising the molecular representation."
    )

print("All available internal fingerprint comparisons passed.")


training_fp_compatibility.to_csv(
    OUT / "internal_fingerprint_structure_compatibility.csv",
    index=False
)


             drug  n_candidate_structures                                    selected_source  selected_bit_difference  minimum_candidate_bit_difference  any_candidate_matches_stored
0   5FLUOROURACIL                       1  local:data/processed/dataset_assembly/drug_smi...                      0.0                               0.0                          True
1     ADAVOSERTIB                       1  local:data/raw/external_validation/hoefer_2025...                      0.0                               0.0                          True
2         AZD7762                       1  local:data/processed/dataset_assembly/drug_smi...                      0.0                               0.0                          True
3       CISPLATIN                       2  local:data/processed/dataset_assembly/drug_smi...                      0.0                               0.0                          True
4      CRIZOTINIB                       1  local:data/raw/external_validation/hoefer_2025.

Internal training drugs: 26
Internal drugs appearing in external data: 24
Internal drugs absent from external data: ['DACTOLISIB', 'IRINOTECAN']
Drugs with comparable source SMILES: 24
Exact fingerprint matches: 24
Drugs without comparable source SMILES: ['DACTOLISIB', 'IRINOTECAN']
All available internal fingerprint comparisons passed.


## 9. Availability of molecular structures for external compounds

In [17]:
# Final analysis uses locally versioned structure sources.
# No external API requests are made during evaluation.

cold_drugs = sorted(
    d for d in external_drugs
    if d not in internal_fp_lookup
)

still_unresolved = sorted(
    d for d in cold_drugs
    if d not in structure_bank
)

print("External cold compounds:", len(cold_drugs))
print("Unresolved compounds:", still_unresolved)

External cold compounds: 211
Unresolved compounds: ['CREBI1', 'DITC', 'OLIGOMYCIN']


## 10. External Morgan fingerprints and coverage assessment

In [18]:
def fp_from_smiles(s):
    mol = Chem.MolFromSmiles(str(s))
    if mol is None:
        return None
    bv = AllChem.GetMorganGenerator(radius=2, fpSize=512).GetFingerprint(mol)
    arr = np.zeros((512,), dtype=np.float32)
    DataStructs.ConvertToNumpyArray(bv, arr)
    return arr

external_fp_cache = {}

def resolve_fp(drug):
    if drug in internal_fp_lookup:
        return internal_fp_lookup[drug], "stored_internal_jaaks_fp"

    if drug in external_fp_cache:
        return external_fp_cache[drug], structure_source.get(
            drug, "cached_external_structure"
        )

    if drug in structure_bank:
        arr = fp_from_smiles(structure_bank[drug])
        if arr is not None:
            external_fp_cache[drug] = arr
            return arr, structure_source.get(drug, "external_structure")

    return None, "unresolved"

feature_rows = []
feature_matrix = []

for r in external_units.itertuples(index=False):
    afp, asrc = resolve_fp(r.drug_a_canon)
    bfp, bsrc = resolve_fp(r.drug_b_canon)

    ok = afp is not None and bfp is not None

    rec = {
        "dataset": r.dataset,
        "pair_key": r.pair_key,
        "drug_a_canon": r.drug_a_canon,
        "drug_b_canon": r.drug_b_canon,
        "canonical_external_subset": r.canonical_external_subset,
        "drug_a_fp_source": asrc,
        "drug_b_fp_source": bsrc,
        "features_ok": ok,
    }

    if ok:
        feat = np.concatenate(
            [(afp + bfp) / 2.0, np.abs(afp - bfp)]
        ).astype(np.float32)

        if len(feat) != 1024:
            raise ValueError("Pair feature length != 1024.")

        rec["_feature_index"] = len(feature_matrix)
        feature_matrix.append(feat)
    else:
        rec["_feature_index"] = np.nan

    feature_rows.append(rec)

feature_audit = pd.DataFrame(feature_rows)

coverage = (
    feature_audit.groupby(
        ["dataset", "canonical_external_subset"],
        as_index=False,
    )
    .agg(
        n_pairs_total=("pair_key", "nunique"),
        n_pairs_scored=("features_ok", "sum"),
    )
)

coverage["coverage_fraction"] = (
    coverage["n_pairs_scored"] / coverage["n_pairs_total"]
)

display(coverage)
display(feature_audit.loc[~feature_audit.features_ok])

feature_audit.to_csv(
    OUT / "external_pair_structure_feature_audit.csv",
    index=False,
)
coverage.to_csv(
    OUT / "external_task_structure_coverage.csv",
    index=False,
)


   dataset  canonical_external_subset  n_pairs_total  n_pairs_scored  coverage_fraction
0    Bashi            both_cold_drugs             58              58           1.000000
1    Bashi              one_cold_drug             43              42           0.976744
2    Bashi  same_pair_reproducibility              8               8           1.000000
3   Hoefer              one_cold_drug            133             133           1.000000
4   Hoefer  same_pair_reproducibility             13              13           1.000000
5    NCATS            both_cold_drugs            465             465           1.000000
6    NCATS              one_cold_drug             31              31           1.000000
7      Vis              one_cold_drug             16              16           1.000000
8      Vis  same_pair_reproducibility             32              32           1.000000
9     Yale            both_cold_drugs            465             406           0.873118
10    Yale              one_cold

     dataset                   pair_key   drug_a_canon drug_b_canon canonical_external_subset                                   drug_a_fp_source                                   drug_b_fp_source  features_ok  \
10      Yale             17DMAG||CREBI1         17DMAG       CREBI1           both_cold_drugs  pubchem_cache:data/processed/external_validati...                                         unresolved        False   
26      Yale         17DMAG||OLIGOMYCIN         17DMAG   OLIGOMYCIN           both_cold_drugs  pubchem_cache:data/processed/external_validati...                                         unresolved        False   
49      Yale      5FLUOROURACIL||CREBI1  5FLUOROURACIL       CREBI1             one_cold_drug                           stored_internal_jaaks_fp                                         unresolved        False   
65      Yale  5FLUOROURACIL||OLIGOMYCIN  5FLUOROURACIL   OLIGOMYCIN             one_cold_drug                           stored_internal_jaaks_fp        

In [19]:
#Experimental registry and fingerprint-coverage checks.

source_size = external_units["dataset"].value_counts().to_dict()

assert source_size == EXPECTED_SOURCE_SIZES, (
    f"Unexpected study counts: {source_size}"
)

expected_subsets = {
    "Yale": {
        "same_pair_reproducibility": 45,
        "one_cold_drug": 310,
        "both_cold_drugs": 465,
    },
    "NCATS": {
        "one_cold_drug": 31,
        "both_cold_drugs": 465,
    },
    "Hoefer": {
        "same_pair_reproducibility": 13,
        "one_cold_drug": 133,
    },
    "Vis": {
        "same_pair_reproducibility": 32,
        "one_cold_drug": 16,
    },
    "Bashi": {
        "same_pair_reproducibility": 8,
        "one_cold_drug": 43,
        "both_cold_drugs": 58,
    },
}

for dataset, expected in expected_subsets.items():
    observed = (
        external_units.loc[
            external_units["dataset"].eq(dataset),
            "canonical_external_subset"
        ]
        .value_counts()
        .to_dict()
    )
    assert observed == expected, (
        f"{dataset}: {observed} != {expected}"
    )

n_complete = int(feature_audit["features_ok"].sum())

assert len(external_units) == 1619
assert n_complete == len(feature_matrix)
assert int(coverage["n_pairs_scored"].sum()) == n_complete

display(
    coverage[
        [
            "dataset",
            "canonical_external_subset",
            "n_pairs_total",
            "n_pairs_scored",
            "coverage_fraction",
        ]
    ]
)

print("External experimental pairs:", len(external_units))
print("Pairs with transferable fingerprints:", n_complete)
print("Pairs excluded for missing fingerprints:",
      len(external_units) - n_complete)

print("External registry and fingerprint checks passed.")


   dataset  canonical_external_subset  n_pairs_total  n_pairs_scored  coverage_fraction
0    Bashi            both_cold_drugs             58              58           1.000000
1    Bashi              one_cold_drug             43              42           0.976744
2    Bashi  same_pair_reproducibility              8               8           1.000000
3   Hoefer              one_cold_drug            133             133           1.000000
4   Hoefer  same_pair_reproducibility             13              13           1.000000
5    NCATS            both_cold_drugs            465             465           1.000000
6    NCATS              one_cold_drug             31              31           1.000000
7      Vis              one_cold_drug             16              16           1.000000
8      Vis  same_pair_reproducibility             32              32           1.000000
9     Yale            both_cold_drugs            465             406           0.873118
10    Yale              one_cold

External experimental pairs: 1619
Pairs with transferable fingerprints: 1539
Pairs excluded for missing fingerprints: 80
External registry and fingerprint checks passed.


In [20]:

unresolved_drugs = sorted(
    set(cold_drugs) - set(structure_bank)
)

missing_rows = feature_audit.loc[
    ~feature_audit["features_ok"]
].copy()

rows = []

for drug in unresolved_drugs:
    affected = missing_rows.loc[
        missing_rows["drug_a_canon"].eq(drug)
        | missing_rows["drug_b_canon"].eq(drug)
    ]

    for dataset, count in affected.groupby("dataset").size().items():
        rows.append({
            "drug": drug,
            "dataset": dataset,
            "affected_pairs": int(count)
        })

unresolved_audit = pd.DataFrame(
    rows,
    columns=["drug", "dataset", "affected_pairs"]
)

display(unresolved_audit)

unresolved_audit.to_csv(
    OUT / "unresolved_compounds_by_dataset.csv",
    index=False
)


         drug dataset  affected_pairs
0      CREBI1    Yale              40
1        DITC   Bashi               1
2  OLIGOMYCIN    Yale              40

## 11. Refit the three model families on the internal training data

HistGradientBoosting, LightGBM and CatBoost are each refitted on the complete 8,450-unit internal training dataset for the any-positive and majority-positive endpoints.

Only the 1,024-dimensional symmetric Morgan-fingerprint representation is used. External experimental responses are excluded from model fitting and are used exclusively for independent evaluation.

The model specifications correspond to the retained configurations documented in the thesis.

In [21]:
models = {}
fit_rows = []

for endpoint, y in TARGETS.items():
    for model_name in MODELS:
        print("FIT", endpoint, model_name)
        m = make_model(model_name, y)
        m.fit(Xtrain, y)
        models[(endpoint, model_name)] = m
        fit_rows.append({
            "model_spec_version": MODEL_SPEC_VERSION,
            "endpoint": endpoint,
            "model": model_name,
            "n_train": len(y),
            "n_positive": int(y.sum()),
            "model_parameters_json": json.dumps(
    m.get_params(),
    sort_keys=True,
    default=str,
),
        })

fit_log = pd.DataFrame(fit_rows)
fit_log.to_csv(OUT / "final_external_model_fit_log.csv", index=False)
display(fit_log)


FIT any_positive_label hist_gradient_boosting
FIT any_positive_label lightgbm_gbdt
FIT any_positive_label catboost
FIT majority_label hist_gradient_boosting
FIT majority_label lightgbm_gbdt
FIT majority_label catboost


                 model_spec_version            endpoint                   model  n_train  n_positive                              model_parameters_json
0  thesis_appendix_B_three_model_v1  any_positive_label  hist_gradient_boosting     8450        1063  {"categorical_features": "from_dtype", "class_...
1  thesis_appendix_B_three_model_v1  any_positive_label           lightgbm_gbdt     8450        1063  {"boosting_type": "gbdt", "class_weight": "bal...
2  thesis_appendix_B_three_model_v1  any_positive_label                catboost     8450        1063  {"allow_writing_files": false, "auto_class_wei...
3  thesis_appendix_B_three_model_v1      majority_label  hist_gradient_boosting     8450         188  {"categorical_features": "from_dtype", "class_...
4  thesis_appendix_B_three_model_v1      majority_label           lightgbm_gbdt     8450         188  {"boosting_type": "gbdt", "class_weight": "bal...
5  thesis_appendix_B_three_model_v1      majority_label                catboost     8450

## 12. Generate external predictions

In [22]:
if not feature_matrix:
    raise RuntimeError("No external pair has a complete Morgan representation.")

FM = np.vstack(feature_matrix).astype(np.float32)

row_by_feature_index = (
    feature_audit.dropna(subset=["_feature_index"])
    .assign(_feature_index=lambda x: x["_feature_index"].astype(int))
    .set_index("_feature_index")
)

pred_rows = []

for endpoint in TARGETS:
    for model_name in MODELS:
        m = models[(endpoint, model_name)]
        pr = m.predict_proba(FM)[:, 1]

        for idx, score in enumerate(pr):
            r = row_by_feature_index.loc[idx]
            pred_rows.append({
                "dataset": r.dataset,
                "pair_key": r.pair_key,
                "canonical_external_subset": r.canonical_external_subset,
                "internal_endpoint": endpoint,
                "model": model_name,
                "model_score": float(score),
            })

predictions = pd.DataFrame(pred_rows)

expected_rows = (
    int(feature_audit.features_ok.sum())
    * len(TARGETS)
    * len(MODELS)
)

if len(predictions) != expected_rows:
    raise ValueError((len(predictions), expected_rows))

print("Final external prediction rows:", len(predictions))
display(predictions.head())


Final external prediction rows: 9234


  dataset               pair_key canonical_external_subset   internal_endpoint                   model  model_score
0    Yale  17DMAG||5FLUOROURACIL             one_cold_drug  any_positive_label  hist_gradient_boosting     0.020159
1    Yale       17DMAG||AFATINIB           both_cold_drugs  any_positive_label  hist_gradient_boosting     0.027841
2    Yale      17DMAG||ALPELISIB           both_cold_drugs  any_positive_label  hist_gradient_boosting     0.018380
3    Yale        17DMAG||AZD1480           both_cold_drugs  any_positive_label  hist_gradient_boosting     0.038775
4    Yale        17DMAG||AZD7762             one_cold_drug  any_positive_label  hist_gradient_boosting     0.087568

In [23]:

# Bridge: freshly rebuilt external units and predictions.
# No historical prediction files are loaded.

final_units = external_units.merge(
    feature_audit[["dataset", "pair_key", "features_ok"]],
    on=["dataset", "pair_key"],
    how="left",
    validate="one_to_one",
)

if final_units["features_ok"].isna().any():
    raise ValueError("Some external units lack a feature audit.")

final_units["features_ok"] = as_bool_series(
    final_units["features_ok"]
).to_numpy()

final_units["external_test_unit_id"] = (
    final_units["dataset"].astype(str)
    + "::"
    + final_units["pair_key"].astype(str)
)

final_predictions = predictions.merge(
    final_units[
        ["dataset", "pair_key", "external_test_unit_id"]
    ],
    on=["dataset", "pair_key"],
    how="left",
    validate="many_to_one",
)

assert len(final_units) == 1619
assert int(final_units["features_ok"].sum()) == n_complete
assert final_predictions["external_test_unit_id"].notna().all()

assert len(final_predictions) == (
    n_complete * len(MODELS) * len(TARGETS)
)

print("External test units:", len(final_units))
print("Feature-complete units:", n_complete)
print("Fresh predictions:", len(final_predictions))


External test units: 1619
Feature-complete units: 1539
Fresh predictions: 9234


In [24]:

#Three-model refit and prediction checks.
# Run after the fresh-prediction bridge cell.

expected_models = set(FINAL_MODEL_PANEL)
expected_endpoints = set(ENDPOINTS)

assert set(MODELS) == expected_models
assert set(TARGETS) == expected_endpoints

# Exactly one fit per model and endpoint.
assert len(fit_log) == (
    len(expected_models) * len(expected_endpoints)
)
assert set(fit_log["model"]) == expected_models
assert set(fit_log["endpoint"]) == expected_endpoints
assert not fit_log.duplicated(["endpoint", "model"]).any()

# External test registry and fingerprint coverage.
assert len(final_units) == 1619
assert (
    final_units["dataset"].value_counts().to_dict()
    == EXPECTED_SOURCE_SIZES
)
assert int(as_bool_series(final_units["features_ok"]).sum()) == n_complete

# One prediction per feature-complete pair, model and endpoint.
expected_predictions = (
    n_complete * len(expected_models) * len(expected_endpoints)
)

assert len(final_predictions) == expected_predictions
assert set(final_predictions["model"]) == expected_models
assert set(final_predictions["internal_endpoint"]) == expected_endpoints
assert final_predictions["model_score"].between(0, 1).all()
assert final_predictions["external_test_unit_id"].notna().all()

assert not final_predictions.duplicated(
    ["external_test_unit_id", "internal_endpoint", "model"]
).any()

print("THREE-MODEL REFIT VALIDATION PASSED")
print("Internal training units:", len(train))
print("External experimental pairs:", len(final_units))
print("Feature-complete pairs:", n_complete)
print("Model fits:", len(fit_log))
print("Fresh prediction rows:", len(final_predictions))


THREE-MODEL REFIT VALIDATION PASSED
Internal training units: 8450
External experimental pairs: 1619
Feature-complete pairs: 1539
Model fits: 6
Fresh prediction rows: 9234


## 13. Three-model rank-percentile consensus

In [25]:
final_units = final_units.copy()
final_units["features_ok"] = as_bool_series(final_units["features_ok"]).to_numpy()

final_predictions=final_predictions.reset_index(drop=True).copy()

assert "external_test_unit_id" in final_units.columns
assert "external_test_unit_id" in final_predictions.columns
assert final_predictions["external_test_unit_id"].notna().all()

final_predictions["model_rank_percentile"]=np.nan
for _,idx in final_predictions.groupby(["dataset","canonical_external_subset","internal_endpoint","model"]).groups.items():
    s=final_predictions.loc[idx,"model_score"]
    n=len(s)
    final_predictions.loc[idx,"model_rank_percentile"] = 1.0 if n<=1 else (s.rank(method="average")-1)/(n-1)
assert final_predictions["model_rank_percentile"].notna().all()

consensus_core=(final_predictions.groupby(["dataset","canonical_external_subset","pair_key","internal_endpoint"],as_index=False)
    .agg(consensus_rank_percentile=("model_rank_percentile","mean"),
         rank_sd=("model_rank_percentile","std"),
         n_models=("model","nunique"),
         mean_raw_probability=("model_score","mean")))
assert (consensus_core["n_models"]==3).all()
assert len(consensus_core) == n_complete * len(ENDPOINTS)

meta=final_units.drop(columns=["internal_endpoint"],errors="ignore")
final_consensus=consensus_core.merge(meta,on=["dataset","canonical_external_subset","pair_key"],how="left",validate="many_to_one")
assert final_consensus["external_test_unit_id"].notna().all()

print("External units:",len(final_units))
print("Feature-complete units:",int(as_bool_series(final_units["features_ok"]).sum()))
print("Prediction rows:",len(final_predictions))
print("Consensus rows:",len(final_consensus))


External units: 1619
Feature-complete units: 1539
Prediction rows: 9234
Consensus rows: 3078


## 14. External performance and chemical-representation sensitivity

In [26]:
def safe_auc(y,s):
    z=pd.DataFrame({"y":y,"s":s}).dropna()
    if len(z)<2 or z["y"].nunique()<2:return np.nan
    return float(roc_auc_score(z["y"],z["s"]))

def safe_ap(y,s):
    z=pd.DataFrame({"y":y,"s":s}).dropna()
    if len(z)<2 or z["y"].nunique()<2:return np.nan
    return float(average_precision_score(z["y"],z["s"]))

def safe_spearman(x,y):
    z=pd.DataFrame({"x":x,"y":y}).dropna()
    if len(z)<4 or z["x"].nunique()<2 or z["y"].nunique()<2:return np.nan,np.nan
    r,p=spearmanr(z["x"],z["y"]);return float(r),float(p)

def bh_qvalues(pvals):
    p=pd.Series(pvals,dtype=float);out=pd.Series(np.nan,index=p.index,dtype=float);valid=p.dropna()
    if valid.empty:return out
    order=valid.sort_values().index; ranked=valid.loc[order].to_numpy();m=len(ranked)
    q=ranked*m/np.arange(1,m+1);q=np.minimum.accumulate(q[::-1])[::-1];q=np.clip(q,0,1)
    out.loc[order]=q;return out


# Source-specific external validation of the three-model
# rank consensus for both internal training endpoints.

metric_rows = []

group_cols = [
    "dataset",
    "canonical_external_subset",
    "internal_endpoint",
]

for (dataset, subset, endpoint), d in final_consensus.groupby(group_cols):

    d = d.loc[d["features_ok"].astype(bool)].copy()

    base = final_units.loc[
        final_units["dataset"].eq(dataset)
        & final_units["canonical_external_subset"].eq(subset)
    ]

    n_total = len(base)
    n_scored = len(d)

    y = pd.to_numeric(
        d["external_binary_pair_any"], errors="coerce"
    )
    response = pd.to_numeric(
        d["external_continuous_higher_is_better"],
        errors="coerce",
    )
    score = d["consensus_rank_percentile"]

    rho, p_value = safe_spearman(score, response)

    metric_rows.append({
        "dataset": dataset,
        "canonical_external_subset": subset,
        "internal_endpoint": endpoint,
        "n_pairs_total": n_total,
        "n_pairs_scored": n_scored,
        "coverage_fraction": n_scored / n_total,
        "binary_prevalence_scored": (
            float(y.mean()) if y.notna().any() else np.nan
        ),
        "consensus_auc": safe_auc(y, score),
        "consensus_ap": safe_ap(y, score),
        "spearman_vs_continuous": rho,
        "spearman_continuous_p": p_value,
        "mean_rank_sd": float(d["rank_sd"].mean()),
    })

metrics = pd.DataFrame(metric_rows)

# BH correction across the external continuous-response
# correlation tests, as specified in thesis Section 2.5.
metrics["spearman_continuous_q_bh"] = bh_qvalues(
    metrics["spearman_continuous_p"]
)

assert len(metrics) == 24

display(
    metrics.sort_values(group_cols)
)


   dataset  canonical_external_subset   internal_endpoint  n_pairs_total  n_pairs_scored  coverage_fraction  binary_prevalence_scored  consensus_auc  consensus_ap  spearman_vs_continuous  spearman_continuous_p  \
0    Bashi            both_cold_drugs  any_positive_label             58              58           1.000000                  0.258621       0.489147      0.248159               -0.166226               0.212369   
1    Bashi            both_cold_drugs      majority_label             58              58           1.000000                  0.258621       0.401550      0.218138               -0.421478               0.000984   
2    Bashi              one_cold_drug  any_positive_label             43              42           0.976744                  0.404762       0.681176      0.552278                0.360122               0.019148   
3    Bashi              one_cold_drug      majority_label             43              42           0.976744                  0.404762       0.505882

In [27]:

# Sensitivity of external predictions to alternative raw SMILES.
# Primary predictions, model fits and reported metrics are unchanged.

alternative_smiles = {}

for drug in form_sensitive_drugs:
    choices = {smi for smi, _ in structure_candidates[drug]}
    alternatives = choices - {structure_bank[drug]}

    if len(alternatives) != 1:
        raise ValueError(
            f"{drug}: expected one alternative, "
            f"found {len(alternatives)}"
        )

    alternative_smiles[drug] = next(iter(alternatives))

# Find scored combinations containing any of the six drugs.
affected = final_units.loc[
    as_bool_series(final_units["features_ok"]).to_numpy()
    & (
        final_units["drug_a_canon"].isin(alternative_smiles)
        | final_units["drug_b_canon"].isin(alternative_smiles)
    )
].copy()

alt_features = []

for r in affected.itertuples(index=False):
    a = (
        fp_from_smiles(alternative_smiles[r.drug_a_canon])
        if r.drug_a_canon in alternative_smiles
        else resolve_fp(r.drug_a_canon)[0]
    )
    b = (
        fp_from_smiles(alternative_smiles[r.drug_b_canon])
        if r.drug_b_canon in alternative_smiles
        else resolve_fp(r.drug_b_canon)[0]
    )

    if a is None or b is None:
        raise ValueError(f"Missing fingerprint: {r.pair_key}")

    alt_features.append(
        np.concatenate([
            (a + b) / 2,
            np.abs(a - b),
        ]).astype(np.float32)
    )

if not alt_features:
    raise ValueError("No affected scored pairs found.")

alt_X = np.vstack(alt_features)

# Score alternative fingerprints with existing fitted models.
patch_rows = []

for (endpoint, model_name), fitted in models.items():
    scores = fitted.predict_proba(alt_X)[:, 1]

    for r, score in zip(affected.itertuples(index=False), scores):
        patch_rows.append({
            "dataset": r.dataset,
            "pair_key": r.pair_key,
            "internal_endpoint": endpoint,
            "model": model_name,
            "alternative_score": float(score),
        })

patch = pd.DataFrame(patch_rows)

keys = ["dataset", "pair_key", "internal_endpoint", "model"]

assert not patch.duplicated(keys).any()

score_changes = patch.merge(
    final_predictions[keys + ["model_score"]],
    on=keys,
    validate="one_to_one",
)

score_changes["absolute_score_change"] = (
    score_changes["alternative_score"]
    - score_changes["model_score"]
).abs()

# Create a separate prediction table with alternatives substituted.
alt_pred = final_predictions.drop(
    columns=["model_rank_percentile"],
    errors="ignore",
).merge(
    patch,
    on=keys,
    how="left",
    validate="one_to_one",
)

alt_pred["model_score"] = alt_pred[
    "alternative_score"
].fillna(alt_pred["model_score"])

alt_pred.drop(columns=["alternative_score"], inplace=True)

# Recalculate ranks across entire source/novelty groups,
# including unchanged pairs.
alt_pred["model_rank_percentile"] = np.nan

rank_groups = [
    "dataset", "canonical_external_subset",
    "internal_endpoint", "model",
]

for _, idx in alt_pred.groupby(rank_groups).groups.items():
    s = alt_pred.loc[idx, "model_score"]
    alt_pred.loc[idx, "model_rank_percentile"] = (
        1.0 if len(s) <= 1
        else (s.rank(method="average") - 1) / (len(s) - 1)
    )

consensus_keys = [
    "dataset", "canonical_external_subset",
    "pair_key", "internal_endpoint",
]

alt_consensus = (
    alt_pred.groupby(consensus_keys, as_index=False)
    .agg(
        consensus_rank_percentile=(
            "model_rank_percentile", "mean"
        )
    )
    .merge(
        final_units,
        on=["dataset", "canonical_external_subset", "pair_key"],
        how="left",
        validate="many_to_one",
    )
)

assert len(alt_consensus) == len(final_consensus)

# Recalculate complete evaluation metrics and BH correction.
metric_keys = [
    "dataset", "canonical_external_subset", "internal_endpoint"
]

alt_rows = []

for grouping, part in alt_consensus.groupby(metric_keys):
    score = part["consensus_rank_percentile"]

    binary = pd.to_numeric(
        part["external_binary_pair_any"], errors="coerce"
    )
    response = pd.to_numeric(
        part["external_continuous_higher_is_better"],
        errors="coerce",
    )

    rho, pval = safe_spearman(score, response)

    alt_rows.append(dict(zip(metric_keys, grouping)) | {
        "alt_auc": safe_auc(binary, score),
        "alt_ap": safe_ap(binary, score),
        "alt_rho": rho,
        "alt_p": pval,
    })

alt_metrics = pd.DataFrame(alt_rows)
alt_metrics["alt_q"] = bh_qvalues(alt_metrics["alt_p"])

comparison = metrics.merge(
    alt_metrics, on=metric_keys, validate="one_to_one"
)

for new_col, original_col in [
    ("alt_auc", "consensus_auc"),
    ("alt_ap", "consensus_ap"),
    ("alt_rho", "spearman_vs_continuous"),
    ("alt_q", "spearman_continuous_q_bh"),
]:
    comparison[f"delta_{new_col}"] = (
        comparison[new_col] - comparison[original_col]
    )

comparison["significance_changed"] = (
    comparison["alt_q"].lt(0.05)
    != comparison["spearman_continuous_q_bh"].lt(0.05)
)

score_changes.to_csv(
    OUT / "chemical_form_sensitivity_scores.csv",
    index=False,
)
comparison.to_csv(
    OUT / "chemical_form_sensitivity_metrics.csv",
    index=False,
)

print("Affected external pairs:", len(affected))
print(
    "Mean absolute probability change:",
    score_changes["absolute_score_change"].mean()
)
print(
    "Maximum absolute probability change:",
    score_changes["absolute_score_change"].max()
)
print(
    "Tasks changing BH significance:",
    int(comparison["significance_changed"].sum())
)

display(comparison[
    metric_keys + [
        "delta_alt_auc", "delta_alt_ap",
        "delta_alt_rho", "delta_alt_q",
        "significance_changed",
    ]
])


Affected external pairs: 65
Mean absolute probability change: 0.005856355038626672
Maximum absolute probability change: 0.07568738713799653
Tasks changing BH significance: 0


   dataset  canonical_external_subset   internal_endpoint  delta_alt_auc  delta_alt_ap  delta_alt_rho  delta_alt_q  significance_changed
0    Bashi            both_cold_drugs  any_positive_label       0.000000      0.000000       0.000000     0.000000                 False
1    Bashi            both_cold_drugs      majority_label       0.000000      0.000000       0.000000     0.000000                 False
2    Bashi              one_cold_drug  any_positive_label       0.000000      0.000000       0.000000     0.000000                 False
3    Bashi              one_cold_drug      majority_label       0.000000      0.000000       0.000000     0.000000                 False
4    Bashi  same_pair_reproducibility  any_positive_label       0.000000      0.000000       0.000000    -0.011664                 False
5    Bashi  same_pair_reproducibility      majority_label       0.000000      0.000000       0.000000     0.000000                 False
6   Hoefer              one_cold_drug  an

In [28]:
# Rebuild the 12-task manifest directly from the authoritative final unit registry.
manifest_group_cols=[c for c in ["dataset","study_role","independence_class","canonical_external_subset"] if c in final_units.columns]
task_manifest=(final_units.groupby(manifest_group_cols,as_index=False)
    .agg(n_pairs=("pair_key","nunique"),n_pairs_with_features=("features_ok",lambda x:int(pd.Series(x).fillna(False).astype(bool).sum())),
         n_binary_nonmissing=("external_binary_pair_any",lambda x:int(pd.Series(x).notna().sum())),
         n_continuous_nonmissing=("external_continuous_higher_is_better",lambda x:int(pd.Series(x).notna().sum()))))
task_manifest["feature_coverage"]=task_manifest["n_pairs_with_features"]/task_manifest["n_pairs"]

assert len(final_units)==1619

assert int(as_bool_series(final_units["features_ok"]).sum()) == n_complete

assert len(final_predictions) == (
    n_complete * len(FINAL_MODEL_PANEL) * len(ENDPOINTS)
)

assert len(final_consensus) == (
    n_complete * len(ENDPOINTS)
)

assert len(task_manifest)==12
assert final_units["dataset"].value_counts().to_dict()==EXPECTED_SOURCE_SIZES
assert set(final_predictions["model"].unique())==FINAL_MODEL_PANEL
assert set(final_predictions["internal_endpoint"].unique())==ENDPOINTS
assert not final_predictions.duplicated(["external_test_unit_id","internal_endpoint","model"]).any()
assert final_predictions["model_score"].between(0,1).all()

novelty_summary=(final_units.groupby(["dataset","canonical_external_subset"],as_index=False)
    .agg(n_pairs=("pair_key","nunique"),
         n_binary=("external_binary_pair_any",lambda x:int(pd.Series(x).notna().sum())),
         n_continuous=("external_continuous_higher_is_better",lambda x:int(pd.Series(x).notna().sum())),
         n_with_features=("features_ok",lambda x:int(pd.Series(x).fillna(False).astype(bool).sum()))))

display(task_manifest)
display(novelty_summary)
print("External validation integrity checks passed.")

   dataset                                         study_role                      independence_class  canonical_external_subset  n_pairs  n_pairs_with_features  n_binary_nonmissing  n_continuous_nonmissing  \
0    Bashi    full-matrix experimental replication / transfer          related_ecosystem_experimental            both_cold_drugs       58                     58                   58                       58   
1    Bashi    full-matrix experimental replication / transfer          related_ecosystem_experimental              one_cold_drug       43                     42                   43                       43   
2    Bashi    full-matrix experimental replication / transfer          related_ecosystem_experimental  same_pair_reproducibility        8                      8                    8                        8   
3   Hoefer  targeted gemcitabine-anchored multi-line PDAC ...       cross_assay_external_experimental              one_cold_drug      133                    133

   dataset  canonical_external_subset  n_pairs  n_binary  n_continuous  n_with_features
0    Bashi            both_cold_drugs       58        58            58               58
1    Bashi              one_cold_drug       43        43            43               42
2    Bashi  same_pair_reproducibility        8         8             8                8
3   Hoefer              one_cold_drug      133       133           133              133
4   Hoefer  same_pair_reproducibility       13        13            13               13
5    NCATS            both_cold_drugs      465       465           465              465
6    NCATS              one_cold_drug       31        31            31               31
7      Vis              one_cold_drug       16         0            16               16
8      Vis  same_pair_reproducibility       32         0            32               32
9     Yale            both_cold_drugs      465         0           465              406
10    Yale              one_cold

External validation integrity checks passed.


## 15. Export external validation results

In [29]:

# Save all tables required to reproduce and report
# the external experimental validation.

exports = {
    "final_external_ml_test_units.csv": final_units,
    "final_external_ml_predictions_three_model.csv":
        final_predictions,
    "final_external_ml_consensus_three_model.csv":
        final_consensus,
    "final_external_ml_task_manifest_three_model.csv":
        task_manifest,
    "final_external_ml_task_metrics_three_model.csv":
        metrics,
    "final_external_novelty_domain_summary.csv":
        novelty_summary,
}

for filename, df in exports.items():
    path = DERIVED_OUT / filename
    df.to_csv(path, index=False)
    print("Saved:", path, df.shape)


Saved: C:\Users\Käyttäjä\Documents\PDAC-drug-combination-ML-MSc-thesis\results\tables\ml_comprehensive\external_experimental_validation\final_external_ml_test_units.csv (1619, 18)
Saved: C:\Users\Käyttäjä\Documents\PDAC-drug-combination-ML-MSc-thesis\results\tables\ml_comprehensive\external_experimental_validation\final_external_ml_predictions_three_model.csv (9234, 8)
Saved: C:\Users\Käyttäjä\Documents\PDAC-drug-combination-ML-MSc-thesis\results\tables\ml_comprehensive\external_experimental_validation\final_external_ml_consensus_three_model.csv (3078, 23)
Saved: C:\Users\Käyttäjä\Documents\PDAC-drug-combination-ML-MSc-thesis\results\tables\ml_comprehensive\external_experimental_validation\final_external_ml_task_manifest_three_model.csv (12, 9)
Saved: C:\Users\Käyttäjä\Documents\PDAC-drug-combination-ML-MSc-thesis\results\tables\ml_comprehensive\external_experimental_validation\final_external_ml_task_metrics_three_model.csv (24, 13)
Saved: C:\Users\Käyttäjä\Documents\PDAC-drug-combinat

## 16. External generalisation figure

In [38]:
summary_cols=["dataset","canonical_external_subset","internal_endpoint","n_pairs_total","n_pairs_scored","coverage_fraction","binary_prevalence_scored","consensus_auc","consensus_ap","spearman_vs_continuous","spearman_continuous_q_bh"]
source_order=["Yale","NCATS","Hoefer","Vis","Bashi"]
source_labels={"Yale":"Yale","NCATS":"NCATS/PANC-1","Hoefer":"Höfer","Vis":"Vis","Bashi":"Bashi"}
subset_order=["same_pair_reproducibility","one_cold_drug","both_cold_drugs"]
subset_labels={"same_pair_reproducibility":"Same pair","one_cold_drug":"One cold","both_cold_drugs":"Both cold"}
endpoint_order=["any_positive_label","majority_label"]
endpoint_titles={"any_positive_label":"Any-positive model","majority_label":"Majority-positive model"}

rho_matrices={};q_matrices={}
for endpoint in endpoint_order:
    rho=pd.DataFrame(np.nan,index=source_order,columns=subset_order);q=rho.copy()
    for _,row in metrics[metrics["internal_endpoint"].eq(endpoint)].iterrows():
        if row["dataset"] in rho.index and row["canonical_external_subset"] in rho.columns:
            rho.loc[row["dataset"],row["canonical_external_subset"]]=row["spearman_vs_continuous"]
            q.loc[row["dataset"],row["canonical_external_subset"]]=row["spearman_continuous_q_bh"]
    rho_matrices[endpoint]=rho;q_matrices[endpoint]=q


fig, axes = plt.subplots(
    1, 2, figsize=(9, 5.2), sharey=True
)

norm = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)
cmap = plt.cm.RdBu.copy()
cmap.set_bad("#E6E6E6")

for ax, endpoint in zip(axes, endpoint_order):

    rho = rho_matrices[endpoint]
    q = q_matrices[endpoint]

    im = ax.imshow(
        np.ma.masked_invalid(rho.to_numpy(float)),
        cmap=cmap,
        norm=norm,
        aspect="auto"
    )

    ax.set_xticks(np.arange(len(subset_order)))
    ax.set_xticklabels(
        [subset_labels[s] for s in subset_order],
        fontsize=10
    )

    ax.set_yticks(np.arange(len(source_order)))
    ax.set_yticklabels(
        [source_labels[s] for s in source_order],
        fontsize=10
    )

    ax.set_title(
        endpoint_titles[endpoint],
        fontsize=11,
        fontweight="bold",
        pad=9
    )

    for i, source in enumerate(source_order):
        for j, subset in enumerate(subset_order):

            rv = rho.loc[source, subset]
            qv = q.loc[source, subset]

            if pd.isna(rv):
                label = "—"
                color = "0.45"
                weight = "normal"
            else:
                sig = pd.notna(qv) and qv < 0.05
                label = f"{rv:.2f}" + ("*" if sig else "")
                color = (
                    "white" if abs(rv) >= 0.45
                    else "black"
                )
                weight = "bold" if sig else "normal"

            ax.text(
                j, i, label,
                ha="center",
                va="center",
                fontsize=11,
                color=color,
                fontweight=weight
            )

    ax.tick_params(axis="both", length=0)


# -------------------------------------------------------
# Set the heatmap layout BEFORE creating the colorbar.
# -------------------------------------------------------

fig.subplots_adjust(
    left=0.18,
    right=0.98,
    top=0.90,
    bottom=0.32,
    wspace=0.16
)

# Get the exact horizontal edges of the heatmaps.
fig.canvas.draw()

left_edge = axes[0].get_position().x0
right_edge = axes[-1].get_position().x1

# Create a dedicated colorbar axis with matching edges.
cax = fig.add_axes([
    left_edge,                # Same left edge as first heatmap
    0.205,                    # Vertical position
    right_edge - left_edge,   # Entire heatmap span
    0.028                     # Colorbar height
])

cbar = fig.colorbar(
    im,
    cax=cax,
    orientation="horizontal"
)

cbar.set_label(
    r"Spearman correlation $r_s$ with external continuous response",
    fontsize=10,
    labelpad=4
)

cbar.ax.tick_params(labelsize=9)

# Footnote below the colorbar.
fig.text(
    0.5,
    0.025,
    r"* Benjamini–Hochberg adjusted $q<0.05$; "
    r"— external novelty category not available",
    ha="center",
    fontsize=9
)

# Confirm the alignment.
assert np.isclose(
    cax.get_position().x0,
    axes[0].get_position().x0
)

assert np.isclose(
    cax.get_position().x1,
    axes[-1].get_position().x1
)

# Save the updated figure.
png = FIG / "final_external_validation_by_novelty_three_model.png"
pdf = FIG / "final_external_validation_by_novelty_three_model.pdf"

fig.savefig(png, dpi=300, bbox_inches="tight")
fig.savefig(pdf, bbox_inches="tight")

plt.show()

print("Saved:", png)
print("Saved:", pdf)



<Figure size 900x520 with 3 Axes>

Saved: C:\Users\Käyttäjä\Documents\PDAC-drug-combination-ML-MSc-thesis\results\figures\ml_comprehensive\external_experimental_validation\final_external_validation_by_novelty_three_model.png
Saved: C:\Users\Käyttäjä\Documents\PDAC-drug-combination-ML-MSc-thesis\results\figures\ml_comprehensive\external_experimental_validation\final_external_validation_by_novelty_three_model.pdf


## 17. Reproducibility and software environment

This section exports a reproducibility manifest containing training-data and experimental-source hashes, molecular-fingerprint settings, fitted model parameters, software versions and output dimensions.

All external predictions are generated from models fitted during the current notebook execution. No previously saved model predictions are required.

In [31]:
#Final reproducibility record.
#No earlier predictions, benchmarks or model outputs are read.

import lightgbm
import catboost

provenance = {
    "analysis": "External experimental validation of PDAC drug-combination models",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),

    "training": {
        "source": rel(TRAIN_PATH),
        "sha256": sha256_file(TRAIN_PATH),
        "n_units": len(train),
        "any_positive_count": int(TARGETS["any_positive_label"].sum()),
        "majority_positive_count": int(TARGETS["majority_label"].sum()),
        "feature_matrix_shape": list(Xtrain.shape),
        "feature_matrix_sha256": hashlib.sha256(
            Xtrain.tobytes()
        ).hexdigest(),
    },

    "experimental_sources": {
        "harmonisation_contract": rel(SOURCE_CONTRACT),
        "contract_sha256": sha256_file(SOURCE_CONTRACT),
        "ncats_source": rel(NCATS_INPUT),
        "ncats_sha256": sha256_file(NCATS_INPUT),
        "n_external_pairs": len(final_units),
        "n_feature_complete_pairs": n_complete,
    },

    "fingerprints": {
        "type": "Morgan",
        "radius": 2,
        "bits_per_drug": 512,
        "use_chirality": False,
        "pair_representation": "mean_plus_absolute_difference",
        "n_pair_features": 1024,
        "structure_sources": structure_provenance,
        "structure_source_locks": STRUCTURE_SOURCE_LOCKS,
    },

    "modelling": {
        "model_specification": MODEL_SPEC_VERSION,
        "model_families": sorted(MODELS),
        "endpoints": sorted(TARGETS),
        "n_fitted_models": len(fit_log),
        "n_prediction_rows": len(final_predictions),
        "n_consensus_rows": len(final_consensus),
        "fitted_parameters": {
            f"{endpoint}::{model_name}": model.get_params()
            for (endpoint, model_name), model in models.items()
        },
    },

    "software": {
        "python": sys.version.split()[0],
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": __import__("scipy").__version__,
        "scikit_learn": sklearn.__version__,
        "lightgbm": lightgbm.__version__,
        "catboost": catboost.__version__,
        "rdkit": rdBase.rdkitVersion,
    },

    "external_response_labels_used_for_training": False,
}

manifest_path = OUT / "external_validation_provenance.json"

manifest_path.write_text(
    json.dumps(
        provenance,
        indent=2,
        ensure_ascii=False,
        default=str,
    ),
    encoding="utf-8",
)

print("External validation complete.")
print("Output directory:", rel(OUT))
print("Provenance manifest:", rel(manifest_path))
print("Experimental pairs:", len(final_units))
print("Feature-complete pairs:", n_complete)
print("Model fits:", len(fit_log))
print("Prediction rows:", len(final_predictions))
print("Consensus rows:", len(final_consensus))


External validation complete.
Output directory: results/tables/ml_comprehensive/external_experimental_validation
Provenance manifest: results/tables/ml_comprehensive/external_experimental_validation/external_validation_provenance.json
Experimental pairs: 1619
Feature-complete pairs: 1539
Model fits: 6
Prediction rows: 9234
Consensus rows: 3078
